### Scatter_unit valid for best window

In [27]:
# ============================================================================
# UPDATED Scatter Plot: Pooled MST+VPS with 5 subsets (SUBSET-SPECIFIC VERSION)
# Color based on significance in EACH SUBSET
# ============================================================================
"""
Creates plots with THREE modalities and FIVE subsets per condition.
Subsets: all, correct, error, high_pdw, low_pdw

KEY METHOD: SUBSET-SPECIFIC RESIDUALIZATION
- Each subset has its own β coefficients (residualized within subset)
- Allows neural code to differ across conditions
- Appropriate for comparing how coding changes with outcome/confidence

COLOR SCHEME:
- Units significant (heading OR choice) in CURRENT subset → modality color
- Units NOT significant in current subset → gray
- MST: solid dots | VPS: hollow dots
- Modality colors: vestibular=black, visual=red, combined=blue
- Darker frame for poster visibility

STATISTICS:
- Calculated for ALL units (not just significant ones)
- Shows how effect magnitudes differ across subsets
- Reveals condition-specific patterns (e.g., error trial sign flip)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
from scipy.stats import pearsonr, spearmanr, ttest_1samp, ttest_rel, linregress
import ast

sys.path.append(r'D:\Neural-Pipeline\source')
from analysis_single_neurons.sliding_partial_corr import PartialCorrelationAnalyzer

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# ============================================================================
# CONFIGURATION
# ============================================================================
subject = 'zarya'
dates = [20250306, 20250411, 20250417, 20250501, 20250523, 20250602, 20250702, 20250710]

# Paths
partialcorr_dir = Path(r'D:\Neural-Pipeline\results\analysis_single_neurons\dot3DMP_partialcorr')
neuralfeatures_dir = Path(r'D:\Neural-Pipeline\results\analysis_single_neurons\dots3DMP_neuralfeatures')

# Neural feature files
regular_RT_file = neuralfeatures_dir / 'zarya_pooled_neural_features_regular_RT_stimOn.csv'
tuning_RT_file = neuralfeatures_dir.parent / 'dots3DMPtuning_neuralfeatures' / 'zarya_pooled_neural_features_tuning_RT_stimOn.csv'
tuning_file = neuralfeatures_dir.parent / 'dots3DMPtuning_neuralfeatures' / 'zarya_pooled_neural_features_tuning_stimOn.csv'

# Time window relative to MEAN RT (ms)
rt_offset_window = [-400, -200]

# Significance threshold
ALPHA = 0.05

# Modalities
modalities = ['ves', 'vis', 'comb']
modality_names = {
    'ves': 'Vestibular',
    'vis': 'Visual',
    'comb': 'Combined'
}

modality_to_condition = {
    'ves': 'mod1_coh1',
    'vis': 'mod2_coh2',
    'comb': 'mod3_coh2'
}

# Modality colors: vestibular=black, visual=red, combined=blue
modality_colors = {
    'ves': 'black',
    'vis': 'red',
    'comb': 'blue'
}

# Subsets (SUBSET-SPECIFIC VERSION)
subset_names = ['all', 'correct', 'error', 'high_pdw', 'low_pdw']
subset_descriptions = {
    'all': 'All small heading trials',
    'correct': 'Correct trials only',
    'error': 'Error trials only',
    'high_pdw': 'High confidence (PDW=1)',
    'low_pdw': 'Low confidence (PDW=0)'
}

print("="*80)
print("SCATTER PLOT: MST+VPS POOLED (SUBSET-SPECIFIC RESIDUALIZATION)")
print("Each subset residualized independently - allows code to differ across conditions")
print("="*80)
print(f"Subject: {subject}")
print(f"Significance threshold: p < {ALPHA}")
print(f"Subsets: {subset_names}")
print(f"MST: solid dots | VPS: hollow dots")
print(f"Color: significant in CURRENT subset")
print("="*80)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def parse_list_column(value):
    """Parse string representation of list to actual list"""
    if isinstance(value, list):
        return value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (ValueError, TypeError):
        pass
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except:
            return []
    return []

def is_selective(row, modality, alpha=0.05):
    """Check if unit is selective for given modality (ANY window)"""
    try:
        p_values = parse_list_column(row[f'anova_{modality}_p'])
        if len(p_values) == 0:
            return False
        return any(not np.isnan(p) and p < alpha for p in p_values)
    except:
        return False

def classify_multisensory_units_from_tuning(tuning_df):
    """Classify multisensory units: (vis AND ves) OR comb"""
    # Parse list columns
    for modality in ['vis', 'ves', 'comb']:
        col = f'anova_{modality}_p'
        if col in tuning_df.columns:
            tuning_df[col] = tuning_df[col].apply(parse_list_column)
    
    # Determine selectivity
    tuning_df['vis_selective'] = tuning_df.apply(
        lambda row: is_selective(row, 'vis', alpha=0.05), axis=1)
    tuning_df['ves_selective'] = tuning_df.apply(
        lambda row: is_selective(row, 'ves', alpha=0.05), axis=1)
    tuning_df['comb_selective'] = tuning_df.apply(
        lambda row: is_selective(row, 'comb', alpha=0.05), axis=1)
    
    # Classify as multisensory
    multisensory_mask = []
    for idx, row in tuning_df.iterrows():
        vis_sel = row['vis_selective']
        ves_sel = row['ves_selective']
        comb_sel = row['comb_selective']
        
        is_multisensory = (vis_sel and ves_sel) or comb_sel
        multisensory_mask.append(is_multisensory)
    
    return np.array(multisensory_mask)

# ============================================================================
# STEP 1: Load Mean RT
# ============================================================================

def load_mean_RT(subject, date):
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    loader = NeuralDataLoader()
    loader.load_session(subject, str(date))
    config = Dots3DMPConfig(subject)
    
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True, cal_mean_RT=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    
    return behavior_converted.get('mean_RT', {})

session_mean_RTs = {}
for date in dates:
    try:
        mean_RT = load_mean_RT(subject, date)
        if mean_RT:
            session_mean_RTs[date] = mean_RT
    except Exception as e:
        print(f"{date}: Failed - {e}")

print(f"\n✓ Loaded mean RT for {len(session_mean_RTs)}/{len(dates)} sessions")

# ============================================================================
# STEP 2: Load Neural Features and Get Multisensory Units
# ============================================================================

print("\nLoading neural features...")
nf_regular_RT = pd.read_csv(regular_RT_file)
nf_tuning_RT = pd.read_csv(tuning_RT_file)
nf_tuning = pd.read_csv(tuning_file)

# Create unit keys
for df in [nf_regular_RT, nf_tuning_RT, nf_tuning]:
    df['unit_key'] = df['subject'].astype(str) + '_' + df['date'].astype(str) + '_' + df['unit_idx'].astype(str)

# Classify multisensory units
print("\nClassifying multisensory units...")
multisensory_mask = classify_multisensory_units_from_tuning(nf_tuning)
multisensory_unit_keys = set(nf_tuning[multisensory_mask]['unit_key'])
print(f"Total multisensory units: {len(multisensory_unit_keys)}")

# Get multisensory units from regular RT data (use as base)
ms_units = nf_regular_RT[nf_regular_RT['unit_key'].isin(multisensory_unit_keys)].copy()
print(f"Multisensory units in regular RT data: {len(ms_units)}")

# ============================================================================
# STEP 3: Load Partial Correlation Data (SUBSET-SPECIFIC VERSION)
# ============================================================================

print("\nLoading partial correlation data (SUBSET-SPECIFIC VERSION)...")
partial_corr_data = {}

for date in dates:
    date_str = str(date)
    
    # Load subset-specific version
    pkl_file = partialcorr_dir / f"{subject}_{date_str}_all_partialcorr_subsetspecific.pkl"
    
    if pkl_file.exists():
        results = PartialCorrelationAnalyzer.load_results(pkl_file)
        partial_corr_data[date] = results
    else:
        # Try old naming convention
        pkl_file_old = partialcorr_dir / f"{subject}_{date_str}_all_partialcorr.pkl"
        if pkl_file_old.exists():
            results = PartialCorrelationAnalyzer.load_results(pkl_file_old)
            partial_corr_data[date] = results

print(f"✓ Loaded {len(partial_corr_data)}/{len(dates)} sessions")

# ============================================================================
# STEP 4: Extract Correlations with P-values (SUBSET-SPECIFIC VERSION)
# ============================================================================

def extract_partialcorr_with_pvalues_subsetspecific(results, date, condition, subset_name, 
                                                     rt_offset_window, mean_RT_dict):
    """
    Extract correlations AND p-values for SUBSET-SPECIFIC version.
    
    Same structure as comparable version, but β coefficients differ per subset.
    """
    alignment = 'stimOn'
    
    if alignment not in results:
        return None, None, None, None, None, None
    
    if condition not in results[alignment]['conditions']:
        return None, None, None, None, None, None
    
    cond_data = results[alignment]['conditions'][condition]
    
    if subset_name not in cond_data:
        return None, None, None, None, None, None
    
    subset_data = cond_data[subset_name]
    
    if condition not in mean_RT_dict:
        return None, None, None, None, None, None
    
    mean_RT_sec = mean_RT_dict[condition]
    
    time_axis_sec = cond_data['time_axis']
    
    target_window_sec = [
        mean_RT_sec + (rt_offset_window[0] / 1000),
        mean_RT_sec + (rt_offset_window[1] / 1000)
    ]
    
    # Extract from subset
    heading_correlations = subset_data['heading_corrs']
    choice_correlations = subset_data['choice_corrs']
    
    heading_pvalues = subset_data.get('heading_pvals', None)
    choice_pvalues = subset_data.get('choice_pvals', None)
    
    unit_ids = results[alignment]['unit_ids']
    unit_areas = results[alignment]['unit_areas']
    
    window_mask = (time_axis_sec >= target_window_sec[0]) & (time_axis_sec <= target_window_sec[1])
    
    if not np.any(window_mask):
        return None, None, None, None, None, None
    
    heading_window = heading_correlations[window_mask, :]
    choice_window = choice_correlations[window_mask, :]
    
    mean_heading_corrs = np.nanmean(heading_window, axis=0)
    mean_choice_corrs = np.nanmean(choice_window, axis=0)
    
    if heading_pvalues is not None and choice_pvalues is not None:
        heading_pval_window = heading_pvalues[window_mask, :]
        choice_pval_window = choice_pvalues[window_mask, :]
        
        min_heading_pvals = np.nanmin(heading_pval_window, axis=0)
        min_choice_pvals = np.nanmin(choice_pval_window, axis=0)
    else:
        min_heading_pvals = np.ones_like(mean_heading_corrs)
        min_choice_pvals = np.ones_like(mean_choice_corrs)
    
    return mean_heading_corrs, mean_choice_corrs, min_heading_pvals, min_choice_pvals, unit_ids, unit_areas

# Extract data for ALL multisensory units, ALL subsets
extracted_data_by_subset = {subset_name: [] for subset_name in subset_names}

print("\nExtracting data for all subsets...")
for subset_name in subset_names:
    print(f"\nProcessing subset: {subset_name}")
    
    for date in dates:
        if date not in partial_corr_data or date not in session_mean_RTs:
            continue
        
        results = partial_corr_data[date]
        mean_RT = session_mean_RTs[date]
        
        for modality in modalities:
            condition = modality_to_condition[modality]
            
            # Get ALL multisensory units for this session
            session_units = ms_units[ms_units['date'] == date]
            
            if len(session_units) == 0:
                continue
            
            result = extract_partialcorr_with_pvalues_subsetspecific(
                results, date, condition, subset_name, rt_offset_window, mean_RT
            )
            
            if result[0] is None:
                continue
            
            heading_corrs, choice_corrs, heading_pvals, choice_pvals, unit_ids, unit_areas = result
            
            for _, unit_row in session_units.iterrows():
                unit_id = unit_row['unit_id']
                
                if unit_id in unit_ids:
                    unit_idx_in_list = unit_ids.index(unit_id)
                    
                    h_corr = heading_corrs[unit_idx_in_list]
                    c_corr = choice_corrs[unit_idx_in_list]
                    h_pval = heading_pvals[unit_idx_in_list]
                    c_pval = choice_pvals[unit_idx_in_list]
                    
                    if not (np.isnan(h_corr) or np.isnan(c_corr)):
                        h_sig = h_pval < ALPHA
                        c_sig = c_pval < ALPHA
                        either_sig = h_sig or c_sig
                        
                        extracted_data_by_subset[subset_name].append({
                            'date': date,
                            'modality': modality,
                            'unit_id': unit_id,
                            'unit_key': unit_row['unit_key'],
                            'area': unit_row['area'],
                            'heading_partial_corr': h_corr,
                            'choice_partial_corr': c_corr,
                            'heading_pval': h_pval,
                            'choice_pval': c_pval,
                            'heading_sig': h_sig,
                            'choice_sig': c_sig,
                            'either_sig': either_sig,
                            'subset_name': subset_name
                        })

# Convert to DataFrames
extracted_dfs = {}
for subset_name in subset_names:
    df = pd.DataFrame(extracted_data_by_subset[subset_name])
    extracted_dfs[subset_name] = df
    print(f"{subset_name}: {len(df)} data points")

# ============================================================================
# STEP 4.5: Define MULTISENSORY population (no additional filtering)
# ============================================================================

print("\n" + "="*80)
print("DEFINING POPULATION: ALL MULTISENSORY UNITS")
print("Each plot will color based on significance IN THAT SUBSET")
print("="*80)

# Verify we have multisensory units only
all_trials_df = extracted_dfs['all']

print(f"Total multisensory units in analysis: {all_trials_df['unit_key'].nunique()}")

# Breakdown by subset
for subset_name in subset_names:
    df = extracted_dfs[subset_name]
    
    n_total = df['unit_key'].nunique()
    n_sig = df[df['either_sig']]['unit_key'].nunique()
    
    # Also count by signal type
    n_h_sig = df[df['heading_sig']]['unit_key'].nunique()
    n_c_sig = df[df['choice_sig']]['unit_key'].nunique()
    n_both = df[df['heading_sig'] & df['choice_sig']]['unit_key'].nunique()
    
    print(f"\n{subset_name}:")
    print(f"  Total units: {n_total}")
    print(f"  Significant: {n_sig} ({n_sig/n_total*100:.1f}%)")
    print(f"    Heading only: {n_h_sig - n_both}")
    print(f"    Choice only: {n_c_sig - n_both}")
    print(f"    Both: {n_both}")

# ============================================================================
# STEP 5: Create Pooled Scatter Plots (Color by CURRENT subset significance)
# ============================================================================

print("\n" + "="*80)
print("STEP 5: Creating Pooled Scatter Plots")
print("ALL plots show same multisensory units")
print("Color indicates significance IN THAT SPECIFIC SUBSET")
print("="*80)

saved_figures = []
nonsig_color = '#CCCCCC'

for subset_name in subset_names:
    extracted_df = extracted_dfs[subset_name]
    
    if len(extracted_df) == 0 or 'area' not in extracted_df.columns:
        continue
    
    # Pool both areas
    pooled_data = extracted_df.copy()
    
    if len(pooled_data) == 0:
        continue
    
    # Count by significance IN THIS SUBSET
    n_total_units = pooled_data['unit_key'].nunique()
    n_sig_units = pooled_data[pooled_data['either_sig']]['unit_key'].nunique()
    
    print(f"\n{'='*60}")
    print(f"Creating plot: MST+VPS POOLED - {subset_name}")
    print(f"Total multisensory units: {n_total_units}")
    print(f"Significant in '{subset_name}': {n_sig_units} ({n_sig_units/n_total_units*100:.1f}%)")
    print(f"{'='*60}")
    
    # Create figure with darker frame
    fig, ax = plt.subplots(1, 1, figsize=(10, 9))
    
    # Make frame darker for poster
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(2.5)
    
    # Calculate global limits (can compute once across all subsets if you want consistency)
    all_h = pooled_data['heading_partial_corr'].values
    all_c = pooled_data['choice_partial_corr'].values
    global_lims = [
        min(np.nanmin(all_h), np.nanmin(all_c)) - 0.05,
        max(np.nanmax(all_h), np.nanmax(all_c)) + 0.05
    ]
    
    # Collect correlation info for subtitle
    corr_strs = []
    
    # Plot each modality
    for modality in modalities:
        mod_data = pooled_data[pooled_data['modality'] == modality]
        
        if len(mod_data) == 0:
            continue
        
        mod_color = modality_colors[modality]
        
        # Separate by significance IN CURRENT SUBSET
        sig_data = mod_data[mod_data['either_sig']]
        nonsig_data = mod_data[~mod_data['either_sig']]
        
        # Further separate by area
        mst_sig = sig_data[sig_data['area'] == 'MST']
        vps_sig = sig_data[sig_data['area'] == 'VPS']
        mst_nonsig = nonsig_data[nonsig_data['area'] == 'MST']
        vps_nonsig = nonsig_data[nonsig_data['area'] == 'VPS']
        
        print(f"  {modality}: Total={len(mod_data)}, "
              f"Sig={len(sig_data)} (MST:{len(mst_sig)}, VPS:{len(vps_sig)})")
        
        # Plot NON-SIGNIFICANT units first (gray, background)
        # Non-significant MST (gray, solid)
        if len(mst_nonsig) > 0:
            ax.scatter(
                mst_nonsig['heading_partial_corr'],
                mst_nonsig['choice_partial_corr'],
                alpha=0.35,
                s=50,
                color=nonsig_color,
                marker='o',
                edgecolors='none',
                zorder=1
            )
        
        # Non-significant VPS (gray, hollow)
        if len(vps_nonsig) > 0:
            ax.scatter(
                vps_nonsig['heading_partial_corr'],
                vps_nonsig['choice_partial_corr'],
                alpha=0.35,
                s=50,
                facecolors='none',
                edgecolors=nonsig_color,
                marker='o',
                linewidth=1.5,
                zorder=1
            )
        
        # Plot SIGNIFICANT units (colored, foreground)
        # Significant MST (colored, solid)
        if len(mst_sig) > 0:
            ax.scatter(
                mst_sig['heading_partial_corr'],
                mst_sig['choice_partial_corr'],
                alpha=0.8,
                s=80,
                color=mod_color,
                marker='o',
                edgecolors='white',
                linewidth=0.5,
                zorder=3
            )
        
        # Significant VPS (colored, hollow)
        if len(vps_sig) > 0:
            ax.scatter(
                vps_sig['heading_partial_corr'],
                vps_sig['choice_partial_corr'],
                alpha=0.9,
                s=80,
                facecolors='none',
                edgecolors=mod_color,
                marker='o',
                linewidth=2.5,
                zorder=3
            )
        
        # Calculate correlation for ALL units (to show overall pattern)
        if len(mod_data) > 2:
            r_all, p_all = pearsonr(
                mod_data['heading_partial_corr'],
                mod_data['choice_partial_corr']
            )
            sig_marker = '***' if p_all < 0.001 else '**' if p_all < 0.01 else '*' if p_all < 0.05 else 'ns'
            corr_strs.append(f'{modality}: r={r_all:.2f}{sig_marker} (n_sig={len(sig_data)})')
            
            # Regression line for ALL units (shows overall trend)
            slope, intercept, _, _, _ = linregress(
                mod_data['heading_partial_corr'],
                mod_data['choice_partial_corr']
            )
            x_fit = np.linspace(global_lims[0], global_lims[1], 100)
            y_fit = slope * x_fit + intercept
            ax.plot(x_fit, y_fit, 
                   color=mod_color,
                   linestyle='-', 
                   linewidth=2.5, 
                   alpha=0.5,
                   zorder=2)
    
    # Reference lines (darker for poster)
    ax.axhline(0, color='gray', linestyle='--', alpha=0.6, linewidth=1.5, zorder=0)
    ax.axvline(0, color='gray', linestyle='--', alpha=0.6, linewidth=1.5, zorder=0)
    ax.plot(global_lims, global_lims, 'k--', alpha=0.5, linewidth=2, zorder=0)
    
    # Labels (larger for poster)
    ax.set_xlabel('Heading Partial Correlation\nR(FR, heading | choice, PDW)', 
                 fontsize=16, fontweight='bold')
    ax.set_ylabel('Choice Partial Correlation\nR(FR, choice | heading, PDW)', 
                 fontsize=16, fontweight='bold')
    
    # Title with correlations
    corr_line = ' | '.join(corr_strs) if corr_strs else 'No data'
    
    ax.set_title(f'MST+VPS: {subset_name.upper()}\n'
                f'{subset_descriptions[subset_name]}\n'
                f'(Subset-specific β, n_total={n_total_units}, n_sig={n_sig_units}) '
                f'{corr_line}',
                fontsize=13, fontweight='bold', pad=15)
    
    # Grid (slightly darker)
    ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.8, zorder=0)
    
    # Tick parameters (larger for poster)
    ax.tick_params(axis='both', which='major', labelsize=12, width=2, length=6)
    
    ax.set_xlim(global_lims)
    ax.set_ylim(global_lims)
    ax.set_aspect('equal', adjustable='box')
    
    plt.tight_layout()
    
    # Save
    save_path = partialcorr_dir / f'{subject}_MST+VPS_allMS_{subset_name}_subsetSpecific_RT{rt_offset_window[0]:+d}to{rt_offset_window[1]:+d}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    saved_figures.append(save_path)
    print(f"  ✓ Saved: {save_path.name}")
    
    plt.close(fig)

# ============================================================================
# STEP 6: Generate Statistics (Report for ALL units + significant subset)
# ============================================================================

print("\n" + "="*80)
print("GENERATING STATISTICS (ALL MULTISENSORY UNITS)")
print("="*80)

all_stats_lines = []
all_stats_lines.append("="*80)
all_stats_lines.append("CORRELATION ANALYSIS: HEADING vs CHOICE (MST+VPS POOLED)")
all_stats_lines.append("SUBSET-SPECIFIC VERSION: Different β for each subset")
all_stats_lines.append("Population: All multisensory units")
all_stats_lines.append("Significance tested independently per subset")
all_stats_lines.append("="*80)
all_stats_lines.append("")

summary_table = []

for subset_name in subset_names:
    extracted_df = extracted_dfs[subset_name]
    
    if len(extracted_df) == 0 or 'area' not in extracted_df.columns:
        continue
    
    all_stats_lines.append("\n" + "="*80)
    all_stats_lines.append(f"SUBSET: {subset_name.upper()}")
    all_stats_lines.append(f"Description: {subset_descriptions[subset_name]}")
    all_stats_lines.append("="*80)
    
    # Pool across areas
    pooled_data = extracted_df.copy()
    
    all_stats_lines.append(f"\nMST+VPS POOLED:")
    all_stats_lines.append(f"{'-'*80}")
    
    for modality in modalities:
        mod_data = pooled_data[pooled_data['modality'] == modality]
        
        if len(mod_data) == 0:
            continue
        
        # Get ALL units and significant subset
        all_units = mod_data
        sig_units = mod_data[mod_data['either_sig']]
        
        # Breakdown by area
        mst_all = all_units[all_units['area'] == 'MST']
        vps_all = all_units[all_units['area'] == 'VPS']
        mst_sig = sig_units[sig_units['area'] == 'MST']
        vps_sig = sig_units[sig_units['area'] == 'VPS']
        
        all_stats_lines.append(f"\n{modality_names[modality]}:")
        all_stats_lines.append(f"  Total units: {len(all_units)} (MST: {len(mst_all)}, VPS: {len(vps_all)})")
        all_stats_lines.append(f"  Significant in '{subset_name}': {len(sig_units)} "
                              f"({len(sig_units)/len(all_units)*100:.1f}%)")
        all_stats_lines.append(f"    MST sig: {len(mst_sig)}, VPS sig: {len(vps_sig)}")
        
        # Breakdown by signal type
        h_sig = sig_units[sig_units['heading_sig']]
        c_sig = sig_units[sig_units['choice_sig']]
        both_sig = sig_units[sig_units['heading_sig'] & sig_units['choice_sig']]
        
        all_stats_lines.append(f"  Signal types:")
        all_stats_lines.append(f"    Heading only: {len(h_sig) - len(both_sig)}")
        all_stats_lines.append(f"    Choice only: {len(c_sig) - len(both_sig)}")
        all_stats_lines.append(f"    Both: {len(both_sig)}")
        
        # Descriptive statistics for ALL units
        h_mean_all = all_units['heading_partial_corr'].mean()
        h_std_all = all_units['heading_partial_corr'].std()
        h_median_all = all_units['heading_partial_corr'].median()
        
        c_mean_all = all_units['choice_partial_corr'].mean()
        c_std_all = all_units['choice_partial_corr'].std()
        c_median_all = all_units['choice_partial_corr'].median()
        
        all_stats_lines.append(f"\n  Heading Partial Correlation (ALL units):")
        all_stats_lines.append(f"    Mean ± SD: {h_mean_all:+.4f} ± {h_std_all:.4f}")
        all_stats_lines.append(f"    Median: {h_median_all:+.4f}")
        all_stats_lines.append(f"    Range: [{all_units['heading_partial_corr'].min():.4f}, "
                              f"{all_units['heading_partial_corr'].max():.4f}]")
        
        all_stats_lines.append(f"\n  Choice Partial Correlation (ALL units):")
        all_stats_lines.append(f"    Mean ± SD: {c_mean_all:+.4f} ± {c_std_all:.4f}")
        all_stats_lines.append(f"    Median: {c_median_all:+.4f}")
        all_stats_lines.append(f"    Range: [{all_units['choice_partial_corr'].min():.4f}, "
                              f"{all_units['choice_partial_corr'].max():.4f}]")
        
        # Descriptive statistics for SIGNIFICANT units
        if len(sig_units) > 0:
            h_mean_sig = sig_units['heading_partial_corr'].mean()
            h_std_sig = sig_units['heading_partial_corr'].std()
            c_mean_sig = sig_units['choice_partial_corr'].mean()
            c_std_sig = sig_units['choice_partial_corr'].std()
            
            all_stats_lines.append(f"\n  Heading Partial Correlation (SIGNIFICANT units only):")
            all_stats_lines.append(f"    Mean ± SD: {h_mean_sig:+.4f} ± {h_std_sig:.4f}")
            
            all_stats_lines.append(f"\n  Choice Partial Correlation (SIGNIFICANT units only):")
            all_stats_lines.append(f"    Mean ± SD: {c_mean_sig:+.4f} ± {c_std_sig:.4f}")
        
        # Correlation between heading and choice (ALL units - KEY ANALYSIS)
        if len(all_units) > 2:
            r_pearson, p_pearson = pearsonr(
                all_units['heading_partial_corr'],
                all_units['choice_partial_corr']
            )
            r_spearman, p_spearman = spearmanr(
                all_units['heading_partial_corr'],
                all_units['choice_partial_corr']
            )
            
            sig_marker_p = '***' if p_pearson < 0.001 else '**' if p_pearson < 0.01 else '*' if p_pearson < 0.05 else 'n.s.'
            sig_marker_s = '***' if p_spearman < 0.001 else '**' if p_spearman < 0.01 else '*' if p_spearman < 0.05 else 'n.s.'
            
            all_stats_lines.append(f"\n  Correlation between Heading and Choice (ALL units):")
            all_stats_lines.append(f"    Pearson r = {r_pearson:+.4f}, p = {p_pearson:.6f} {sig_marker_p}")
            all_stats_lines.append(f"    Spearman rho = {r_spearman:+.4f}, p = {p_spearman:.6f} {sig_marker_s}")
            
            # Store for summary table
            summary_table.append({
                'subset_name': subset_name,
                'area': 'MST+VPS',
                'modality': modality_names[modality],
                'n_total': len(all_units),
                'n_sig': len(sig_units),
                'n_mst': len(mst_all),
                'n_vps': len(vps_all),
                'n_mst_sig': len(mst_sig),
                'n_vps_sig': len(vps_sig),
                'pct_sig': len(sig_units)/len(all_units)*100,
                'heading_mean_all': h_mean_all,
                'heading_std_all': h_std_all,
                'choice_mean_all': c_mean_all,
                'choice_std_all': c_std_all,
                'heading_mean_sig': h_mean_sig if len(sig_units) > 0 else np.nan,
                'choice_mean_sig': c_mean_sig if len(sig_units) > 0 else np.nan,
                'pearson_r': r_pearson,
                'pearson_p': p_pearson,
                'spearman_r': r_spearman,
                'spearman_p': p_spearman
            })
        
        # Test if mean correlations differ from zero (ALL units)
        if len(all_units) > 2:
            t_h, p_h = ttest_1samp(all_units['heading_partial_corr'], 0)
            t_c, p_c = ttest_1samp(all_units['choice_partial_corr'], 0)
            
            sig_h = '***' if p_h < 0.001 else '**' if p_h < 0.01 else '*' if p_h < 0.05 else 'n.s.'
            sig_c = '***' if p_c < 0.001 else '**' if p_c < 0.01 else '*' if p_c < 0.05 else 'n.s.'
            
            all_stats_lines.append(f"\n  Test against zero (one-sample t-test, ALL units):")
            all_stats_lines.append(f"    Heading: t({len(all_units)-1}) = {t_h:.4f}, p = {p_h:.6f} {sig_h}")
            all_stats_lines.append(f"    Choice: t({len(all_units)-1}) = {t_c:.4f}, p = {p_c:.6f} {sig_c}")
        
        # Quadrant analysis (ALL units)
        q1 = all_units[(all_units['heading_partial_corr'] > 0) & (all_units['choice_partial_corr'] > 0)]
        q2 = all_units[(all_units['heading_partial_corr'] < 0) & (all_units['choice_partial_corr'] > 0)]
        q3 = all_units[(all_units['heading_partial_corr'] < 0) & (all_units['choice_partial_corr'] < 0)]
        q4 = all_units[(all_units['heading_partial_corr'] > 0) & (all_units['choice_partial_corr'] < 0)]
        
        all_stats_lines.append(f"\n  Quadrant Distribution (ALL units):")
        all_stats_lines.append(f"    Q1 (H+, C+): {len(q1)} ({len(q1)/len(all_units)*100:.1f}%)")
        all_stats_lines.append(f"    Q2 (H-, C+): {len(q2)} ({len(q2)/len(all_units)*100:.1f}%)")
        all_stats_lines.append(f"    Q3 (H-, C-): {len(q3)} ({len(q3)/len(all_units)*100:.1f}%)")
        all_stats_lines.append(f"    Q4 (H+, C-): {len(q4)} ({len(q4)/len(all_units)*100:.1f}%)")

# ============================================================================
# STEP 7: Summary Tables and Cross-Subset Comparisons
# ============================================================================

all_stats_lines.append("\n\n" + "="*80)
all_stats_lines.append("SUMMARY TABLE: CORRELATION COEFFICIENTS (ALL UNITS)")
all_stats_lines.append("="*80)
all_stats_lines.append("")

if len(summary_table) > 0:
    summary_df = pd.DataFrame(summary_table)
    
    # Pivot table for Pearson r
    all_stats_lines.append("Pearson Correlation (r) between Heading and Choice:")
    all_stats_lines.append("-" * 80)
    
    # Header
    all_stats_lines.append(f"{'Subset':<20} {'Ves':<15} {'Vis':<15} {'Comb':<15}")
    all_stats_lines.append("-" * 70)
    
    for subset_name in subset_names:
        subset_data = summary_df[summary_df['subset_name'] == subset_name]
        
        if len(subset_data) == 0:
            continue
        
        row_str = f"{subset_name:<20}"
        
        for mod in ['Vestibular', 'Visual', 'Combined']:
            mod_data = subset_data[subset_data['modality'] == mod]
            
            if len(mod_data) > 0:
                r = mod_data.iloc[0]['pearson_r']
                p = mod_data.iloc[0]['pearson_p']
                sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
                row_str += f"{r:+.3f}{sig:<4} "
            else:
                row_str += f"{'N/A':<15}"
        
        all_stats_lines.append(row_str)
    
    # Mean effect sizes (ALL units)
    all_stats_lines.append("\n\nMean Heading Partial Correlation (ALL units):")
    all_stats_lines.append("-" * 80)
    all_stats_lines.append(f"{'Subset':<20} {'Ves':<15} {'Vis':<15} {'Comb':<15}")
    all_stats_lines.append("-" * 70)
    
    for subset_name in subset_names:
        subset_data = summary_df[summary_df['subset_name'] == subset_name]
        
        if len(subset_data) == 0:
            continue
        
        row_str = f"{subset_name:<20}"
        
        for mod in ['Vestibular', 'Visual', 'Combined']:
            mod_data = subset_data[subset_data['modality'] == mod]
            
            if len(mod_data) > 0:
                mean = mod_data.iloc[0]['heading_mean_all']
                row_str += f"{mean:+.4f}      "
            else:
                row_str += f"{'N/A':<15}"
        
        all_stats_lines.append(row_str)
    
    all_stats_lines.append("\n\nMean Choice Partial Correlation (ALL units):")
    all_stats_lines.append("-" * 80)
    all_stats_lines.append(f"{'Subset':<20} {'Ves':<15} {'Vis':<15} {'Comb':<15}")
    all_stats_lines.append("-" * 70)
    
    for subset_name in subset_names:
        subset_data = summary_df[summary_df['subset_name'] == subset_name]
        
        if len(subset_data) == 0:
            continue
        
        row_str = f"{subset_name:<20}"
        
        for mod in ['Vestibular', 'Visual', 'Combined']:
            mod_data = subset_data[subset_data['modality'] == mod]
            
            if len(mod_data) > 0:
                mean = mod_data.iloc[0]['choice_mean_all']
                row_str += f"{mean:+.4f}      "
            else:
                row_str += f"{'N/A':<15}"
        
        all_stats_lines.append(row_str)
    
    # Percentage significant
    all_stats_lines.append("\n\nPercentage Significant per Subset:")
    all_stats_lines.append("-" * 80)
    all_stats_lines.append(f"{'Subset':<20} {'Ves':<15} {'Vis':<15} {'Comb':<15}")
    all_stats_lines.append("-" * 70)
    
    for subset_name in subset_names:
        subset_data = summary_df[summary_df['subset_name'] == subset_name]
        
        if len(subset_data) == 0:
            continue
        
        row_str = f"{subset_name:<20}"
        
        for mod in ['Vestibular', 'Visual', 'Combined']:
            mod_data = subset_data[subset_data['modality'] == mod]
            
            if len(mod_data) > 0:
                pct = mod_data.iloc[0]['pct_sig']
                n_sig = mod_data.iloc[0]['n_sig']
                n_tot = mod_data.iloc[0]['n_total']
                row_str += f"{pct:.1f}% ({n_sig}/{n_tot})  "
            else:
                row_str += f"{'N/A':<15}"
        
        all_stats_lines.append(row_str)
    
    # Sample size table
    all_stats_lines.append("\n\n" + "="*80)
    all_stats_lines.append("SAMPLE SIZES (Significant / Total)")
    all_stats_lines.append("="*80)
    
    all_stats_lines.append(f"{'Subset':<20} {'Ves':<30} {'Vis':<30} {'Comb':<30}")
    all_stats_lines.append("-" * 110)
    
    for subset_name in subset_names:
        subset_data = summary_df[summary_df['subset_name'] == subset_name]
        
        if len(subset_data) == 0:
            continue
        
        row_str = f"{subset_name:<20}"
        
        for mod in ['Vestibular', 'Visual', 'Combined']:
            mod_data = subset_data[subset_data['modality'] == mod]
            
            if len(mod_data) > 0:
                n_sig = mod_data.iloc[0]['n_sig']
                n_tot = mod_data.iloc[0]['n_total']
                n_mst_sig = mod_data.iloc[0]['n_mst_sig']
                n_vps_sig = mod_data.iloc[0]['n_vps_sig']
                row_str += f"{n_sig}/{n_tot} (M:{n_mst_sig},V:{n_vps_sig})    "
            else:
                row_str += f"{'N/A':<30}"
        
        all_stats_lines.append(row_str)
    
    # Save summary CSV
    summary_csv_path = partialcorr_dir / f'{subject}_MST+VPS_correlation_summary_subsetSpecific.csv'
    summary_df.to_csv(summary_csv_path, index=False)
    print(f"\n✓ Summary CSV saved: {summary_csv_path.name}")

# ============================================================================
# STEP 8: Cross-Subset Comparisons
# ============================================================================

all_stats_lines.append("\n\n" + "="*80)
all_stats_lines.append("CROSS-SUBSET COMPARISONS (SAME MULTISENSORY UNITS)")
all_stats_lines.append("How do correlations differ across conditions?")
all_stats_lines.append("="*80)

if len(summary_table) > 0:
    summary_df = pd.DataFrame(summary_table)
    
    for mod in ['Vestibular', 'Visual', 'Combined']:
        mod_data = summary_df[summary_df['modality'] == mod]
        
        if len(mod_data) < 2:
            continue
        
        all_stats_lines.append(f"\n{mod}:")
        all_stats_lines.append("-" * 60)
        
        # Compare correct vs error
        if 'correct' in mod_data['subset_name'].values and 'error' in mod_data['subset_name'].values:
            corr_data = mod_data[mod_data['subset_name'] == 'correct'].iloc[0]
            err_data = mod_data[mod_data['subset_name'] == 'error'].iloc[0]
            
            h_diff = err_data['heading_mean_all'] - corr_data['heading_mean_all']
            c_diff = err_data['choice_mean_all'] - corr_data['choice_mean_all']
            
            all_stats_lines.append(f"\nCorrect vs Error:")
            all_stats_lines.append(f"  Heading: {corr_data['heading_mean_all']:+.4f} (correct) → "
                                  f"{err_data['heading_mean_all']:+.4f} (error) [Δ={h_diff:+.4f}]")
            all_stats_lines.append(f"  Choice: {corr_data['choice_mean_all']:+.4f} (correct) → "
                                  f"{err_data['choice_mean_all']:+.4f} (error) [Δ={c_diff:+.4f}]")
            all_stats_lines.append(f"  r(H,C): {corr_data['pearson_r']:+.4f} (correct) → "
                                  f"{err_data['pearson_r']:+.4f} (error)")
            all_stats_lines.append(f"  % sig: {corr_data['pct_sig']:.1f}% (correct) → "
                                  f"{err_data['pct_sig']:.1f}% (error)")
        
        # Compare high vs low PDW
        if 'high_pdw' in mod_data['subset_name'].values and 'low_pdw' in mod_data['subset_name'].values:
            high_data = mod_data[mod_data['subset_name'] == 'high_pdw'].iloc[0]
            low_data = mod_data[mod_data['subset_name'] == 'low_pdw'].iloc[0]
            
            h_diff = high_data['heading_mean_all'] - low_data['heading_mean_all']
            c_diff = high_data['choice_mean_all'] - low_data['choice_mean_all']
            
            all_stats_lines.append(f"\nHigh vs Low Confidence:")
            all_stats_lines.append(f"  Heading: {low_data['heading_mean_all']:+.4f} (low) → "
                                  f"{high_data['heading_mean_all']:+.4f} (high) [Δ={h_diff:+.4f}]")
            all_stats_lines.append(f"  Choice: {low_data['choice_mean_all']:+.4f} (low) → "
                                  f"{high_data['choice_mean_all']:+.4f} (high) [Δ={c_diff:+.4f}]")
            all_stats_lines.append(f"  r(H,C): {low_data['pearson_r']:+.4f} (low) → "
                                  f"{high_data['pearson_r']:+.4f} (high)")
            all_stats_lines.append(f"  % sig: {low_data['pct_sig']:.1f}% (low) → "
                                  f"{high_data['pct_sig']:.1f}% (high)")

# Save full statistics text file
stats_file = partialcorr_dir / f'{subject}_MST+VPS_statistics_detailed_subsetSpecific.txt'
with open(stats_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(all_stats_lines))

print(f"✓ Detailed statistics saved: {stats_file.name}")

print("\n" + "="*80)
print("✅ ALL ANALYSES COMPLETE!")
print("="*80)
print(f"\nGenerated {len(saved_figures)} figures")
print(f"Statistics saved to: {stats_file.name}")
if len(summary_table) > 0:
    print(f"Summary table saved to: {summary_csv_path.name}")
print("\nKey Method:")
print("  ✓ Population: ALL multisensory units (consistent)")
print("  ✓ Significance: Tested independently per subset")
print("  ✓ Color: Based on significance IN THAT SUBSET")
print("  ✓ Subset-specific β: Allows code to differ")
print("  ✓ Shows both ALL units and significant subsets")
print("="*80)

SCATTER PLOT: MST+VPS POOLED (SUBSET-SPECIFIC RESIDUALIZATION)
Each subset residualized independently - allows code to differ across conditions
Subject: zarya
Significance threshold: p < 0.05
Subsets: ['all', 'correct', 'error', 'high_pdw', 'low_pdw']
MST: solid dots | VPS: hollow dots
Color: significant in CURRENT subset
Loaded dots3DMP data: zarya20250306dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250306dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Loaded dots3DMP data: zarya20250411dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250411dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Loaded dots3DMP data: zarya20250417dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250417dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Loaded dots3DMP data: zarya20250501dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250501dots3DMPtuning_processed.npz
Loaded dots3DMP c

### Scatter_by units

In [20]:
# ============================================================================
# UPDATED Scatter Plot: Area-based plots with significance and tuning categories
# ============================================================================
"""
Creates area plots (filled regions) with THREE modalities in ONE plot per area.
- Uses combined win green color scheme
- Dots: solid for tuning RT only valid, hollow for both RT types valid
- Modality colors: vestibular=black, visual=red, combined=blue
- Significance: colored if significant for EITHER heading OR choice, gray otherwise
- No legend, statistical text outside figure
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
from scipy.stats import pearsonr, spearmanr, ttest_1samp, ttest_rel
import ast

sys.path.append(r'D:\Neural-Pipeline\source')
from analysis_single_neurons.sliding_partial_corr import PartialCorrelationAnalyzer

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# ============================================================================
# CONFIGURATION
# ============================================================================
subject = 'zarya'
dates = [20250306, 20250411, 20250417, 20250501, 20250523, 20250602, 20250702, 20250710]

# Paths
partialcorr_dir = Path(r'D:\Neural-Pipeline\results\analysis_single_neurons\dot3DMP_partialcorr')
neuralfeatures_dir = Path(r'D:\Neural-Pipeline\results\analysis_single_neurons\dots3DMP_neuralfeatures')

# Neural feature files
regular_RT_file = neuralfeatures_dir / 'zarya_pooled_neural_features_regular_RT_stimOn.csv'
tuning_RT_file = neuralfeatures_dir.parent / 'dots3DMPtuning_neuralfeatures' / 'zarya_pooled_neural_features_tuning_RT_stimOn.csv'
tuning_file = neuralfeatures_dir.parent / 'dots3DMPtuning_neuralfeatures' / 'zarya_pooled_neural_features_tuning_stimOn.csv'

# Time window relative to MEAN RT (ms)
rt_offset_window = [-400, -200]

# Significance threshold
ALPHA = 0.05

# Modalities
modalities = ['ves', 'vis', 'comb']
modality_names = {
    'ves': 'Vestibular',
    'vis': 'Visual',
    'comb': 'Combined'
}

modality_to_condition = {
    'ves': 'mod1_coh1',
    'vis': 'mod2_coh2',
    'comb': 'mod3_coh2'
}

# Modality colors: vestibular=black, visual=red, combined=blue
modality_colors = {
    'ves': 'black',
    'vis': 'red',
    'comb': 'blue'
}

# Areas to plot
areas_to_plot = ['MST', 'VPS']

# Filter types
filter_types = ['small_headings', 'high_pdw_small', 'errors_zeros', 'low_pdw_small']
filter_descriptions = {
    'small_headings': 'All trials with small headings (-10° < h < 10°)',
    'high_pdw_small': 'High PDW (confident) + small headings',
    'errors_zeros': 'Error trials + zero heading trials',
    'low_pdw_small': 'Low PDW (unconfident) + small headings'
}

print("="*80)
print("SCATTER PLOT WITH AREA-BASED ORGANIZATION")
print("="*80)
print(f"Subject: {subject}")
print(f"Significance threshold: p < {ALPHA}")
print(f"Filter types: {filter_types}")
print(f"Areas: {areas_to_plot}")
print("="*80)

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def parse_list_column(value):
    """Parse string representation of list to actual list"""
    if isinstance(value, list):
        return value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except (ValueError, TypeError):
        pass
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except:
            return []
    return []

def is_selective(row, modality, alpha=0.05):
    """Check if unit is selective for given modality (ANY window)"""
    try:
        p_values = parse_list_column(row[f'anova_{modality}_p'])
        if len(p_values) == 0:
            return False
        return any(not np.isnan(p) and p < alpha for p in p_values)
    except:
        return False

def classify_multisensory_units_from_tuning(tuning_df):
    """Classify multisensory units: (vis AND ves) OR comb"""
    # Parse list columns
    for modality in ['vis', 'ves', 'comb']:
        col = f'anova_{modality}_p'
        if col in tuning_df.columns:
            tuning_df[col] = tuning_df[col].apply(parse_list_column)
    
    # Determine selectivity
    tuning_df['vis_selective'] = tuning_df.apply(
        lambda row: is_selective(row, 'vis', alpha=0.05), axis=1)
    tuning_df['ves_selective'] = tuning_df.apply(
        lambda row: is_selective(row, 'ves', alpha=0.05), axis=1)
    tuning_df['comb_selective'] = tuning_df.apply(
        lambda row: is_selective(row, 'comb', alpha=0.05), axis=1)
    
    # Classify as multisensory
    multisensory_mask = []
    for idx, row in tuning_df.iterrows():
        vis_sel = row['vis_selective']
        ves_sel = row['ves_selective']
        comb_sel = row['comb_selective']
        
        is_multisensory = (vis_sel and ves_sel) or comb_sel
        multisensory_mask.append(is_multisensory)
    
    return np.array(multisensory_mask)

def has_valid_tuning(row, modality, prefix='neurometric'):
    """Check if unit has valid tuning"""
    threshold_col = f'{prefix}_{modality}_thresholds'
    r2_col = f'{prefix}_{modality}_r2'
    
    if threshold_col not in row.index or r2_col not in row.index:
        return False
    
    try:
        threshold = row[threshold_col]
        r2 = row[r2_col]
        
        if isinstance(threshold, str):
            threshold = ast.literal_eval(threshold)
            threshold = threshold[0] if len(threshold) > 0 else np.nan
        
        if isinstance(r2, str):
            r2 = ast.literal_eval(r2)
            r2 = r2[0] if len(r2) > 0 else np.nan
        
        return (not np.isnan(threshold) and 
                not np.isinf(threshold) and 
                threshold > 0 and 
                threshold < 100 and
                r2 > 0.5)
    except:
        return False

# ============================================================================
# STEP 1: Load Mean RT
# ============================================================================

def load_mean_RT(subject, date):
    sys.path.append(r'D:\Neural-Pipeline\source')
    from analysis_utils.NeuralDataLoader import NeuralDataLoader, Dots3DMPConfig
    
    loader = NeuralDataLoader()
    loader.load_session(subject, str(date))
    config = Dots3DMPConfig(subject)
    
    behavior_dots3DMP = loader.get_behavioral_data(task='dots3DMP', good_trials_only=True, cal_mean_RT=True)
    behavior_converted = config.convert_behavioral_data(behavior_dots3DMP, task='dots3DMP')
    
    return behavior_converted.get('mean_RT', {})

session_mean_RTs = {}
for date in dates:
    try:
        mean_RT = load_mean_RT(subject, date)
        if mean_RT:
            session_mean_RTs[date] = mean_RT
    except Exception as e:
        print(f"{date}: Failed - {e}")

print(f"\n✓ Loaded mean RT for {len(session_mean_RTs)}/{len(dates)} sessions")

# ============================================================================
# STEP 2: Load Neural Features and Classify Units
# ============================================================================

print("\nLoading neural features...")
nf_regular_RT = pd.read_csv(regular_RT_file)
nf_tuning_RT = pd.read_csv(tuning_RT_file)
nf_tuning = pd.read_csv(tuning_file)

# Create unit keys
for df in [nf_regular_RT, nf_tuning_RT, nf_tuning]:
    df['unit_key'] = df['subject'].astype(str) + '_' + df['date'].astype(str) + '_' + df['unit_idx'].astype(str)

# Classify multisensory units
print("\nClassifying multisensory units...")
multisensory_mask = classify_multisensory_units_from_tuning(nf_tuning)
multisensory_unit_keys = set(nf_tuning[multisensory_mask]['unit_key'])
print(f"Total multisensory units: {len(multisensory_unit_keys)}")

# Categorize units by RT validity
def categorize_units_by_rt(modality, multisensory_keys):
    """Categorize units: tuning RT only, both RT types, or neither"""
    
    # Filter to multisensory units only
    regular_ms = nf_regular_RT[nf_regular_RT['unit_key'].isin(multisensory_keys)].copy()
    tuning_ms = nf_tuning_RT[nf_tuning_RT['unit_key'].isin(multisensory_keys)].copy()
    
    # Check validity
    valid_regular = regular_ms.apply(lambda row: has_valid_tuning(row, modality), axis=1)
    valid_tuning = tuning_ms.apply(lambda row: has_valid_tuning(row, modality), axis=1)
    
    # Create dictionaries
    regular_valid_dict = {}
    for _, row in regular_ms[valid_regular].iterrows():
        regular_valid_dict[row['unit_key']] = True
    
    tuning_valid_dict = {}
    for _, row in tuning_ms[valid_tuning].iterrows():
        tuning_valid_dict[row['unit_key']] = True
    
    # Assign categories
    categorized = regular_ms.copy()
    categorized['valid_regular_RT'] = categorized['unit_key'].map(regular_valid_dict).fillna(False)
    categorized['valid_tuning_RT'] = categorized['unit_key'].map(tuning_valid_dict).fillna(False)
    
    def assign_category(row):
        has_regular = row['valid_regular_RT']
        has_tuning = row['valid_tuning_RT']
        
        if has_tuning and has_regular:
            return 'both'
        elif has_tuning and not has_regular:
            return 'tuning_RT_only'
        else:
            return 'none'
    
    categorized['tuning_category'] = categorized.apply(assign_category, axis=1)
    
    # Keep only valid units
    valid_categories = ['both', 'tuning_RT_only']
    categorized = categorized[categorized['tuning_category'].isin(valid_categories)]
    
    return categorized

categorized_by_modality = {}
for modality in modalities:
    categorized_by_modality[modality] = categorize_units_by_rt(modality, multisensory_unit_keys)
    print(f"{modality}: {len(categorized_by_modality[modality])} valid units")

# ============================================================================
# STEP 3: Load Partial Correlation Data
# ============================================================================

print("\nLoading partial correlation data...")
partial_corr_data = {filter_type: {} for filter_type in filter_types}

for date in dates:
    date_str = str(date)
    
    for filter_type in filter_types:
        pkl_file = partialcorr_dir / f"{subject}_{date_str}_all_partialcorr_centered_{filter_type}.pkl"
        
        if pkl_file.exists():
            results = PartialCorrelationAnalyzer.load_results(pkl_file)
            partial_corr_data[filter_type][date] = results

for filter_type in filter_types:
    n_loaded = len(partial_corr_data[filter_type])
    print(f"✓ {filter_type}: Loaded {n_loaded}/{len(dates)} sessions")

# ============================================================================
# STEP 4: Extract Correlations with P-values
# ============================================================================

def extract_partialcorr_with_pvalues(results, date, condition, rt_offset_window, mean_RT_dict):
    """Extract correlations AND p-values"""
    alignment = 'stimOn'
    
    if alignment not in results:
        return None, None, None, None, None, None
    
    if condition not in results[alignment]['conditions']:
        return None, None, None, None, None, None
    
    if condition not in mean_RT_dict:
        return None, None, None, None, None, None
    
    mean_RT_sec = mean_RT_dict[condition]
    cond_data = results[alignment]['conditions'][condition]
    
    time_axis_sec = cond_data['time_axis']
    
    target_window_sec = [
        mean_RT_sec + (rt_offset_window[0] / 1000),
        mean_RT_sec + (rt_offset_window[1] / 1000)
    ]
    
    heading_correlations = cond_data['heading_corrs']
    choice_correlations = cond_data['choice_corrs']
    
    heading_pvalues = cond_data.get('heading_pvals', None)
    choice_pvalues = cond_data.get('choice_pvals', None)
    
    unit_ids = results[alignment]['unit_ids']
    unit_areas = results[alignment]['unit_areas']
    
    window_mask = (time_axis_sec >= target_window_sec[0]) & (time_axis_sec <= target_window_sec[1])
    
    if not np.any(window_mask):
        return None, None, None, None, None, None
    
    heading_window = heading_correlations[window_mask, :]
    choice_window = choice_correlations[window_mask, :]
    
    mean_heading_corrs = np.nanmean(heading_window, axis=0)
    mean_choice_corrs = np.nanmean(choice_window, axis=0)
    
    if heading_pvalues is not None and choice_pvalues is not None:
        heading_pval_window = heading_pvalues[window_mask, :]
        choice_pval_window = choice_pvalues[window_mask, :]
        
        min_heading_pvals = np.nanmin(heading_pval_window, axis=0)
        min_choice_pvals = np.nanmin(choice_pval_window, axis=0)
    else:
        min_heading_pvals = np.ones_like(mean_heading_corrs)
        min_choice_pvals = np.ones_like(mean_choice_corrs)
    
    return mean_heading_corrs, mean_choice_corrs, min_heading_pvals, min_choice_pvals, unit_ids, unit_areas

# Extract data
extracted_data_by_filter = {filter_type: [] for filter_type in filter_types}

for filter_type in filter_types:
    print(f"\nProcessing filter: {filter_type}")
    
    for date in dates:
        if date not in partial_corr_data[filter_type] or date not in session_mean_RTs:
            continue
        
        results = partial_corr_data[filter_type][date]
        mean_RT = session_mean_RTs[date]
        
        for modality in modalities:
            condition = modality_to_condition[modality]
            
            categorized_units = categorized_by_modality[modality]
            session_units = categorized_units[categorized_units['date'] == date]
            
            if len(session_units) == 0:
                continue
            
            result = extract_partialcorr_with_pvalues(
                results, date, condition, rt_offset_window, mean_RT
            )
            
            if result[0] is None:
                continue
            
            heading_corrs, choice_corrs, heading_pvals, choice_pvals, unit_ids, unit_areas = result
            
            for _, unit_row in session_units.iterrows():
                unit_id = unit_row['unit_id']
                
                if unit_id in unit_ids:
                    unit_idx_in_list = unit_ids.index(unit_id)
                    
                    h_corr = heading_corrs[unit_idx_in_list]
                    c_corr = choice_corrs[unit_idx_in_list]
                    h_pval = heading_pvals[unit_idx_in_list]
                    c_pval = choice_pvals[unit_idx_in_list]
                    
                    if not (np.isnan(h_corr) or np.isnan(c_corr)):
                        h_sig = h_pval < ALPHA
                        c_sig = c_pval < ALPHA
                        either_sig = h_sig or c_sig
                        
                        extracted_data_by_filter[filter_type].append({
                            'date': date,
                            'modality': modality,
                            'unit_id': unit_id,
                            'area': unit_row['area'],
                            'tuning_category': unit_row['tuning_category'],
                            'heading_partial_corr': h_corr,
                            'choice_partial_corr': c_corr,
                            'heading_pval': h_pval,
                            'choice_pval': c_pval,
                            'heading_sig': h_sig,
                            'choice_sig': c_sig,
                            'either_sig': either_sig,
                            'filter_type': filter_type
                        })

# Convert to DataFrames
extracted_dfs = {}
for filter_type in filter_types:
    df = pd.DataFrame(extracted_data_by_filter[filter_type])
    extracted_dfs[filter_type] = df

# ============================================================================
# STEP 5: Create Area-Based Scatter Plots (3 modalities per plot)
# ============================================================================

print("\n" + "="*80)
print("STEP 5: Creating Area-Based Scatter Plots")
print("="*80)

saved_figures = []

# Marker styles: solid for tuning_RT_only, hollow for both
def get_marker_style(category):
    if category == 'tuning_RT_only':
        return 'o', True  # filled circle
    else:  # both
        return 'o', False  # hollow circle

nonsig_color = '#CCCCCC'

for area in areas_to_plot:
    for filter_type in filter_types:
        extracted_df = extracted_dfs[filter_type]
        
        if len(extracted_df) == 0 or 'area' not in extracted_df.columns:
            continue
        
        area_data = extracted_df[extracted_df['area'] == area]
        
        if len(area_data) == 0:
            continue
        
        print(f"\n{'='*60}")
        print(f"Creating plot: {area} - {filter_type}")
        print(f"{'='*60}")
        
        # Create figure with single subplot
        fig, ax = plt.subplots(1, 1, figsize=(10, 9))
        
        # Calculate global limits
        all_h = area_data['heading_partial_corr'].values
        all_c = area_data['choice_partial_corr'].values
        global_lims = [
            min(np.nanmin(all_h), np.nanmin(all_c)) - 0.05,
            max(np.nanmax(all_h), np.nanmax(all_c)) + 0.05
        ]
        
        # Plot each modality
        for modality in modalities:
            mod_data = area_data[area_data['modality'] == modality]
            
            if len(mod_data) == 0:
                continue
            
            mod_color = modality_colors[modality]
            
            # Separate significant and non-significant
            sig_data = mod_data[mod_data['either_sig']]
            nonsig_data = mod_data[~mod_data['either_sig']]
            
            print(f"  {modality}: Total={len(mod_data)}, Sig={len(sig_data)}")
            
            # Plot non-significant (gray)
            if len(nonsig_data) > 0:
                ax.scatter(
                    nonsig_data['heading_partial_corr'],
                    nonsig_data['choice_partial_corr'],
                    alpha=0.3,
                    s=40,
                    color=nonsig_color,
                    marker='x',
                    linewidth=1,
                    zorder=1
                )
            
            # Plot significant by category
            for category in ['tuning_RT_only', 'both']:
                cat_sig_data = sig_data[sig_data['tuning_category'] == category]
                
                if len(cat_sig_data) == 0:
                    continue
                
                marker, filled = get_marker_style(category)
                
                if filled:
                    # Solid dots
                    ax.scatter(
                        cat_sig_data['heading_partial_corr'],
                        cat_sig_data['choice_partial_corr'],
                        alpha=0.8,
                        s=80,
                        color=mod_color,
                        marker=marker,
                        edgecolors='white',
                        linewidth=1,
                        zorder=3
                    )
                else:
                    # Hollow dots
                    ax.scatter(
                        cat_sig_data['heading_partial_corr'],
                        cat_sig_data['choice_partial_corr'],
                        alpha=0.8,
                        s=80,
                        facecolors='none',
                        edgecolors=mod_color,
                        marker=marker,
                        linewidth=2,
                        zorder=3
                    )
        
        # Reference lines
        ax.axhline(0, color='gray', linestyle='--', alpha=0.4, linewidth=1, zorder=0)
        ax.axvline(0, color='gray', linestyle='--', alpha=0.4, linewidth=1, zorder=0)
        ax.plot(global_lims, global_lims, 'k--', alpha=0.3, linewidth=1.5, zorder=0)
        
        # Labels
        ax.set_xlabel('Heading Partial Correlation\nR(FR, heading | choice, PDW)', 
                     fontsize=14, fontweight='bold')
        ax.set_ylabel('Choice Partial Correlation\nR(FR, choice | heading, PDW)', 
                     fontsize=14, fontweight='bold')
        
        ax.set_title(f'{area}: {filter_type.upper()}\n'
                    f'{filter_descriptions[filter_type]}',
                    fontsize=14, fontweight='bold')
        
        ax.grid(True, alpha=0.2, linestyle=':', linewidth=0.5, zorder=0)
        ax.set_xlim(global_lims)
        ax.set_ylim(global_lims)
        ax.set_aspect('equal', adjustable='box')
        
        plt.tight_layout()
        
        # Save
        save_path = partialcorr_dir / f'{subject}_{area}_combined_{filter_type}_RT{rt_offset_window[0]:+d}to{rt_offset_window[1]:+d}.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        saved_figures.append(save_path)
        print(f"  ✓ Saved: {save_path.name}")
        
        plt.close(fig)

# ============================================================================
# STEP 6: Generate Statistics Text Files
# ============================================================================

print("\n" + "="*80)
print("GENERATING STATISTICS")
print("="*80)

for filter_type in filter_types:
    extracted_df = extracted_dfs[filter_type]
    
    if len(extracted_df) == 0 or 'area' not in extracted_df.columns:
        continue
    
    stats_lines = []
    stats_lines.append("="*80)
    stats_lines.append(f"STATISTICS: {filter_type.upper()}")
    stats_lines.append(f"Description: {filter_descriptions[filter_type]}")
    stats_lines.append("="*80)
    stats_lines.append("")
    
    for area in areas_to_plot:
        area_data = extracted_df[extracted_df['area'] == area]
        
        if len(area_data) == 0:
            continue
        
        stats_lines.append(f"\n{area}")
        stats_lines.append("-"*70)
        
        for modality in modalities:
            mod_data = area_data[area_data['modality'] == modality]
            
            if len(mod_data) == 0:
                continue
            
            sig_data = mod_data[mod_data['either_sig']]
            
            stats_lines.append(f"\n{modality_names[modality]}:")
            stats_lines.append(f"  Total units: {len(mod_data)}")
            stats_lines.append(f"  Significant (either): {len(sig_data)} ({len(sig_data)/len(mod_data)*100:.1f}%)")
            
            if len(sig_data) > 0:
                h_mean = sig_data['heading_partial_corr'].mean()
                h_std = sig_data['heading_partial_corr'].std()
                c_mean = sig_data['choice_partial_corr'].mean()
                c_std = sig_data['choice_partial_corr'].std()
                
                stats_lines.append(f"  Heading corr: {h_mean:+.4f} ± {h_std:.4f}")
                stats_lines.append(f"  Choice corr:  {c_mean:+.4f} ± {c_std:.4f}")
                
                if len(sig_data) > 2:
                    r_sig, p_sig = pearsonr(
                        sig_data['heading_partial_corr'],
                        sig_data['choice_partial_corr']
                    )
                    sig_marker = '***' if p_sig < 0.001 else '**' if p_sig < 0.01 else '*' if p_sig < 0.05 else 'n.s.'
                    stats_lines.append(f"  Correlation: r = {r_sig:+.4f}, p = {p_sig:.6f} {sig_marker}")
                
                # By category
                for category in ['tuning_RT_only', 'both']:
                    cat_data = sig_data[sig_data['tuning_category'] == category]
                    if len(cat_data) > 0:
                        cat_label = 'Tuning RT only' if category == 'tuning_RT_only' else 'Both RT types'
                        stats_lines.append(f"    {cat_label}: n={len(cat_data)}")
    
    # Save statistics to file
    stats_file = partialcorr_dir / f'{subject}_statistics_{filter_type}.txt'
    with open(stats_file, 'w') as f:
        f.write('\n'.join(stats_lines))
    
    print(f"✓ Saved statistics: {stats_file.name}")

print("\n" + "="*80)
print("✅ ALL ANALYSES COMPLETE!")
print("="*80)
print(f"\nGenerated {len(saved_figures)} figures")

SCATTER PLOT WITH AREA-BASED ORGANIZATION
Subject: zarya
Significance threshold: p < 0.05
Filter types: ['small_headings', 'high_pdw_small', 'errors_zeros', 'low_pdw_small']
Areas: ['MST', 'VPS']
Loaded dots3DMP data: zarya20250306dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250306dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Loaded dots3DMP data: zarya20250411dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250411dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Loaded dots3DMP data: zarya20250417dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250417dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Loaded dots3DMP data: zarya20250501dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250501dots3DMPtuning_processed.npz
Loaded dots3DMP configuration for subject: zarya
Loaded dots3DMP data: zarya20250523dots3DMP_processed.npz
Loaded dots3DMPtuning data: zarya20250

### heading, stimulus correlaiton sliding window

In [12]:
# ============================================================================
# Sliding Window: Heading vs Choice Correlation by Subpopulation
# MULTIPLE TRIAL FILTERING CONDITIONS - MARKER + ERROR BAR STYLE
# ============================================================================
"""
For each session and filter type, compute the correlation between heading and 
choice partial correlations across the population at each timepoint.

Organization:
- 2 areas × 4 filter types = 8 figures total
- Each figure: 3×3 panels (rows=modalities, cols=alignments)
- 4 lines per panel (4 subpopulations with different markers)

Filter types:
1. small_headings: -10° < heading < 10°
2. high_pdw_small: high PDW + small headings
3. errors_zeros: error trials + zero heading trials
4. low_pdw_small: low PDW + small headings

STYLE: Markers with error bars (no line + shaded region)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
from scipy.stats import pearsonr, spearmanr

sys.path.append(r'D:\Neural-Pipeline\source')
from analysis_single_neurons.sliding_partial_corr import PartialCorrelationAnalyzer

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# ============================================================================
# CONFIGURATION
# ============================================================================
subject = 'zarya'
dates = [20250306, 20250411, 20250417, 20250501, 20250523, 20250602, 20250702, 20250710]

# Paths
partialcorr_dir = Path(r'D:\Neural-Pipeline\results\analysis_single_neurons\dot3DMP_partialcorr')
neuralfeatures_dir = Path(r'D:\Neural-Pipeline\results\analysis_single_neurons\dots3DMP_neuralfeatures')

# Neural feature files
regular_RT_file = neuralfeatures_dir / 'zarya_pooled_neural_features_regular_RT_stimOn.csv'
tuning_RT_file = neuralfeatures_dir.parent / 'dots3DMPtuning_neuralfeatures' / 'zarya_pooled_neural_features_tuning_RT_stimOn.csv'
regular_file = neuralfeatures_dir / 'zarya_pooled_neural_features_regular_saccOnset.csv'

# Modalities and alignments
modalities = ['ves', 'vis', 'comb']
modality_names = {
    'ves': 'Vestibular',
    'vis': 'Visual High Coh',
    'comb': 'Combined High Coh'
}
modality_to_condition = {
    'ves': 'mod1_coh1',
    'vis': 'mod2_coh2',
    'comb': 'mod3_coh2'
}

alignments = ['stimOn', 'saccOnset', 'postTargHold']
alignment_names = {
    'stimOn': 'Stimulus Onset',
    'saccOnset': 'Saccade Onset',
    'postTargHold': 'Post-Saccade'
}

areas_to_plot = ['MST', 'VPS']

# *** NEW: Define filter types ***
filter_types = ['small_headings', 'high_pdw_small', 'errors_zeros', 'low_pdw_small']
filter_descriptions = {
    'small_headings': 'All trials with small headings (-10° < h < 10°)',
    'high_pdw_small': 'High PDW (confident) + small headings',
    'errors_zeros': 'Error trials + zero heading trials',
    'low_pdw_small': 'Low PDW (unconfident) + small headings'
}

# Subpopulation info - NOW WITH MARKERS
subpop_info = {
    'tuning_RT_only': {
        'label': 'Tuning RT Only',
        'color': '#3498DB',
        'marker': 'o',
        'markersize': 6,
        'alpha': 0.8
    },
    'regular_RT_only': {
        'label': 'Regular RT Only',
        'color': '#2ECC71',
        'marker': 's',
        'markersize': 6,
        'alpha': 0.8
    },
    'both': {
        'label': 'Both',
        'color': '#E74C3C',
        'marker': '^',
        'markersize': 7,
        'alpha': 0.8
    },
    'best_window_only': {
        'label': 'Best Window Only',
        'color': '#95A5A6',
        'marker': 'D',
        'markersize': 5,
        'alpha': 0.8
    }
}

print("="*80)
print("SLIDING WINDOW CORRELATION ANALYSIS - BY SUBPOPULATION")
print("MULTIPLE TRIAL FILTERING CONDITIONS")
print("STYLE: MARKERS + ERROR BARS")
print("="*80)
print(f"Subject: {subject}")
print(f"Sessions: {len(dates)}")
print(f"Filter types: {filter_types}")
print(f"Areas: {areas_to_plot}")
print("="*80)

# ============================================================================
# STEP 1: Load Neural Feature Properties and Categorize
# ============================================================================
print("\n" + "="*80)
print("STEP 1: Loading Neural Features and Categorizing Units")
print("="*80)

nf_regular_RT = pd.read_csv(regular_RT_file)
nf_tuning_RT = pd.read_csv(tuning_RT_file)
nf_regular = pd.read_csv(regular_file)

print(f"Loaded regular_RT: {len(nf_regular_RT)} units")
print(f"Loaded tuning_RT: {len(nf_tuning_RT)} units")
print(f"Loaded regular (best window): {len(nf_regular)} units")

def has_valid_tuning(row, modality, prefix='neurometric'):
    """Check if unit has valid tuning"""
    threshold_col = f'{prefix}_{modality}_thresholds'
    r2_col = f'{prefix}_{modality}_r2'
    
    if threshold_col not in row.index or r2_col not in row.index:
        return False
    
    try:
        threshold = row[threshold_col]
        r2 = row[r2_col]
        
        if isinstance(threshold, str):
            import ast
            threshold = ast.literal_eval(threshold)
            threshold = threshold[0] if len(threshold) > 0 else np.nan
        
        if isinstance(r2, str):
            import ast
            r2 = ast.literal_eval(r2)
            r2 = r2[0] if len(r2) > 0 else np.nan
        
        return (not np.isnan(threshold) and 
                not np.isinf(threshold) and 
                threshold > 0 and 
                threshold < 100 and
                r2 > 0.5)
    except:
        return False

def categorize_units(modality):
    """Categorize units into 4 subpopulations"""
    print(f"\n  Categorizing for {modality}...")
    
    valid_regular_RT = nf_regular_RT.apply(lambda row: has_valid_tuning(row, modality), axis=1)
    valid_tuning_RT = nf_tuning_RT.apply(lambda row: has_valid_tuning(row, modality), axis=1)
    valid_regular = nf_regular.apply(lambda row: has_valid_tuning(row, modality), axis=1)
    
    categorized = nf_regular_RT.copy()
    categorized['valid_regular_RT'] = valid_regular_RT
    
    tuning_valid_dict = {}
    for _, row in nf_tuning_RT[valid_tuning_RT].iterrows():
        key = (row['date'], row['unit_id'])
        tuning_valid_dict[key] = True
    
    categorized['valid_tuning_RT'] = categorized.apply(
        lambda row: tuning_valid_dict.get((row['date'], row['unit_id']), False), 
        axis=1
    )
    
    regular_valid_dict = {}
    for _, row in nf_regular[valid_regular].iterrows():
        key = (row['date'], row['unit_id'])
        regular_valid_dict[key] = True
    
    categorized['valid_best_window'] = categorized.apply(
        lambda row: regular_valid_dict.get((row['date'], row['unit_id']), False), 
        axis=1
    )
    
    def assign_category(row):
        has_regular_RT = row['valid_regular_RT']
        has_tuning_RT = row['valid_tuning_RT']
        has_best_window = row['valid_best_window']
        
        if has_regular_RT and has_tuning_RT:
            return 'both'
        elif has_regular_RT and not has_tuning_RT:
            return 'regular_RT_only'
        elif has_tuning_RT and not has_regular_RT:
            return 'tuning_RT_only'
        elif has_best_window:
            return 'best_window_only'
        else:
            return 'none'
    
    categorized['tuning_category'] = categorized.apply(assign_category, axis=1)
    
    valid_categories = ['both', 'regular_RT_only', 'tuning_RT_only', 'best_window_only']
    categorized = categorized[categorized['tuning_category'].isin(valid_categories)]
    
    print(f"    Total valid units: {len(categorized)}")
    for cat in valid_categories:
        count = len(categorized[categorized['tuning_category'] == cat])
        print(f"      {cat}: {count}")
    
    return categorized

# Categorize for each modality
categorized_by_modality = {}
for modality in modalities:
    categorized_by_modality[modality] = categorize_units(modality)

# ============================================================================
# STEP 2: Load Partial Correlation Data (ALL FILTER TYPES)
# ============================================================================
print("\n" + "="*80)
print("STEP 2: Loading Partial Correlation Data (All Filter Types)")
print("="*80)

partial_corr_data = {filter_type: {} for filter_type in filter_types}

for date in dates:
    date_str = str(date)
    
    for filter_type in filter_types:
        pkl_file = partialcorr_dir / f"{subject}_{date_str}_all_partialcorr_centered_{filter_type}.pkl"
        
        if pkl_file.exists():
            results = PartialCorrelationAnalyzer.load_results(pkl_file)
            partial_corr_data[filter_type][date] = results
            print(f"{date_str} - {filter_type}: ✓")
        else:
            print(f"{date_str} - {filter_type}: ⚠ File not found")

for filter_type in filter_types:
    n_loaded = len(partial_corr_data[filter_type])
    print(f"\n✓ {filter_type}: Loaded {n_loaded}/{len(dates)} sessions")

# ============================================================================
# STEP 3: Extract Sliding Window Correlations (FOR EACH FILTER TYPE)
# ============================================================================
print("\n" + "="*80)
print("STEP 3: Computing Sliding Window Correlations")
print("="*80)

def compute_sliding_correlations_for_session(date, alignment, modality, area, categorized_units, filter_data):
    """Compute correlations for one session"""
    if date not in filter_data:
        return None, None
    
    results = filter_data[date]
    
    if alignment not in results:
        return None, None
    
    condition = modality_to_condition[modality]
    if condition not in results[alignment]['conditions']:
        return None, None
    
    cond_data = results[alignment]['conditions'][condition]
    
    time_axis = cond_data['time_axis']
    n_times = len(time_axis)
    
    heading_corrs_all_time = cond_data['heading_corrs']
    choice_corrs_all_time = cond_data['choice_corrs']
    
    unit_ids = results[alignment]['unit_ids']
    unit_areas = results[alignment]['unit_areas']
    
    session_units = categorized_units[categorized_units['date'] == date]
    session_units = session_units[session_units['area'] == area]
    
    if len(session_units) == 0:
        return None, None
    
    units_by_category = {cat: [] for cat in ['tuning_RT_only', 'regular_RT_only', 'both', 'best_window_only']}
    
    for _, unit_row in session_units.iterrows():
        unit_id = unit_row['unit_id']
        category = unit_row['tuning_category']
        
        if unit_id in unit_ids:
            unit_idx = unit_ids.index(unit_id)
            units_by_category[category].append(unit_idx)
    
    correlations = {}
    
    for category, unit_indices in units_by_category.items():
        if len(unit_indices) < 3:
            correlations[category] = np.full(n_times, np.nan)
            continue
        
        corrs_over_time = np.zeros(n_times)
        
        for t in range(n_times):
            heading_t = heading_corrs_all_time[t, unit_indices]
            choice_t = choice_corrs_all_time[t, unit_indices]
            
            valid = ~(np.isnan(heading_t) | np.isnan(choice_t))
            
            if np.sum(valid) >= 3:
                r, _ = pearsonr(heading_t[valid], choice_t[valid])
                corrs_over_time[t] = r
            else:
                corrs_over_time[t] = np.nan
        
        correlations[category] = corrs_over_time
    
    return correlations, time_axis

# Collect correlations for ALL filter types
all_correlations = {
    filter_type: {
        area: {
            alignment: {
                modality: {
                    category: [] for category in ['tuning_RT_only', 'regular_RT_only', 'both', 'best_window_only']
                } for modality in modalities
            } for alignment in alignments
        } for area in areas_to_plot
    } for filter_type in filter_types
}

# Store time axes per filter type, alignment, and modality
time_axes_master = {
    filter_type: {
        alignment: {
            modality: None for modality in modalities
        } for alignment in alignments
    } for filter_type in filter_types
}

print("\nProcessing sessions:")
for filter_type in filter_types:
    print(f"\n{'='*60}")
    print(f"Filter: {filter_type}")
    print(f"{'='*60}")
    
    filter_data = partial_corr_data[filter_type]
    
    for date in dates:
        if date not in filter_data:
            print(f"{date}: Skipped (no data)")
            continue
        
        print(f"{date}:")
        
        for alignment in alignments:
            for modality in modalities:
                categorized = categorized_by_modality[modality]
                
                for area in areas_to_plot:
                    corrs, time_axis = compute_sliding_correlations_for_session(
                        date, alignment, modality, area, categorized, filter_data
                    )
                    
                    if corrs is not None:
                        if time_axes_master[filter_type][alignment][modality] is None:
                            time_axes_master[filter_type][alignment][modality] = time_axis
                        
                        for category, corr_array in corrs.items():
                            all_correlations[filter_type][area][alignment][modality][category].append(corr_array)
                        
                        n_valid = sum([not np.all(np.isnan(c)) for c in corrs.values()])
                        print(f"  {alignment}/{modality}/{area}: {n_valid}/4 categories")

# ============================================================================
# STEP 4: Compute Statistics (FOR EACH FILTER TYPE)
# ============================================================================
print("\n" + "="*80)
print("STEP 4: Computing Statistics Across Sessions")
print("="*80)

def compute_mean_sem(corr_list):
    """Compute mean and SEM using Fisher Z transformation"""
    if len(corr_list) == 0:
        return None, None
    
    corr_array = np.array(corr_list)
    all_nan_mask = np.all(np.isnan(corr_array), axis=0)
    
    if np.all(all_nan_mask):
        return None, None
    
    with np.errstate(invalid='ignore'):
        z_array = np.arctanh(corr_array)
    
    mean_z = np.nanmean(z_array, axis=0)
    n_valid = np.sum(~np.isnan(z_array), axis=0)
    sem_z = np.nanstd(z_array, axis=0) / np.sqrt(n_valid)
    
    with np.errstate(invalid='ignore'):
        mean_r = np.tanh(mean_z)
        sem_lower = np.tanh(mean_z - sem_z)
        sem_upper = np.tanh(mean_z + sem_z)
    
    sem_r = (sem_upper - sem_lower) / 2
    
    return mean_r, sem_r

stats = {
    filter_type: {
        area: {
            alignment: {
                modality: {
                    category: {'mean': None, 'sem': None, 'n_sessions': 0}
                    for category in ['tuning_RT_only', 'regular_RT_only', 'both', 'best_window_only']
                } for modality in modalities
            } for alignment in alignments
        } for area in areas_to_plot
    } for filter_type in filter_types
}

for filter_type in filter_types:
    print(f"\n{filter_type}:")
    for area in areas_to_plot:
        print(f"  {area}:")
        for alignment in alignments:
            for modality in modalities:
                for category in ['tuning_RT_only', 'regular_RT_only', 'both', 'best_window_only']:
                    corr_list = all_correlations[filter_type][area][alignment][modality][category]
                    
                    if len(corr_list) > 0:
                        mean, sem = compute_mean_sem(corr_list)
                        stats[filter_type][area][alignment][modality][category]['mean'] = mean
                        stats[filter_type][area][alignment][modality][category]['sem'] = sem
                        stats[filter_type][area][alignment][modality][category]['n_sessions'] = len(corr_list)
                        
                        if mean is not None:
                            print(f"    {alignment}/{modality}/{category}: {len(corr_list)} sessions")

# ============================================================================
# STEP 5: Create Plots (ONE FIGURE PER FILTER TYPE PER AREA)
# ============================================================================
print("\n" + "="*80)
print("STEP 5: Creating Sliding Window Plots")
print("="*80)

saved_figures = []

for area in areas_to_plot:
    for filter_type in filter_types:
        print(f"\nCreating plot: {area} - {filter_type}...")
        
        fig, axes = plt.subplots(3, 3, figsize=(20, 15))
        
        for row, modality in enumerate(modalities):
            for col, alignment in enumerate(alignments):
                ax = axes[row, col]
                
                time_axis = time_axes_master[filter_type][alignment][modality]
                if time_axis is None:
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                           transform=ax.transAxes, fontsize=14, color='gray')
                    if row == 0:
                        ax.set_title(f'{alignment_names[alignment]}', fontsize=12, fontweight='bold')
                    continue
                
                time_ms = time_axis * 1000
                
                has_data = False
                for category in ['tuning_RT_only', 'regular_RT_only', 'both', 'best_window_only']:
                    mean = stats[filter_type][area][alignment][modality][category]['mean']
                    sem = stats[filter_type][area][alignment][modality][category]['sem']
                    n_sess = stats[filter_type][area][alignment][modality][category]['n_sessions']
                    
                    if mean is None or n_sess == 0:
                        continue
                    
                    has_data = True
                    info = subpop_info[category]
                    
                    ax.errorbar(time_ms, mean, yerr=sem,
                               color=info['color'],
                               marker=info['marker'],
                               markersize=info['markersize'],
                               markerfacecolor=info['color'],
                               markeredgecolor='white',
                               markeredgewidth=0.5,
                               linestyle='none',
                               linewidth=0,
                               capsize=3,
                               capthick=1.5,
                               elinewidth=1.5,
                               label=f"{info['label']} (n={n_sess})",
                               alpha=info['alpha'],
                               zorder=3)
                
                if not has_data:
                    ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                           transform=ax.transAxes, fontsize=14, color='gray')
                
                ax.axhline(0, color='gray', linestyle='--', alpha=0.4, linewidth=1, zorder=1)
                ax.axvline(0, color='gray', linestyle='-', alpha=0.4, linewidth=1.5, zorder=1)
                
                if alignment == 'stimOn' and has_data:
                    ax.text(0.02, 0.98, 't=0: mean RT', transform=ax.transAxes,
                           fontsize=9, va='top', ha='left',
                           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7, edgecolor='gray'))
                
                if row == 2:
                    ax.set_xlabel('Time (ms)', fontsize=11, fontweight='bold')
                if col == 0:
                    ax.set_ylabel('Heading-Choice Correlation (r)', fontsize=11, fontweight='bold')
                
                if row == 0:
                    ax.set_title(f'{alignment_names[alignment]}', fontsize=13, fontweight='bold')
                
                if col == 2:
                    ax.text(1.05, 0.5, modality_names[modality], 
                           transform=ax.transAxes, rotation=270,
                           fontsize=13, fontweight='bold', va='center')
                
                if row == 0 and col == 2 and has_data:
                    ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
                
                ax.grid(True, alpha=0.25, linestyle=':', linewidth=0.5)
                ax.set_ylim([-1.0, 0.4])
                ax.tick_params(labelsize=10)
        
        plt.suptitle(f'{area}: Sliding Window Correlation - {filter_type.upper()}\n'
                     f'{filter_descriptions[filter_type]}',
                    fontsize=16, fontweight='bold', y=0.995)
        
        plt.tight_layout()
        
        save_path = partialcorr_dir / f'{subject}_{area}_sliding_{filter_type}_markers.png'
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        saved_figures.append(save_path)
        print(f"  ✓ Saved: {save_path.name}")
        
        plt.close(fig)

print("\n" + "="*80)
print("✅ SLIDING WINDOW ANALYSIS COMPLETE!")
print("="*80)
print(f"\nGenerated {len(saved_figures)} sliding window plots:")
for fig_path in saved_figures:
    print(f"  - {fig_path.name}")
print("\nStyle: Markers with error bars")
print("Filters: 4 different trial selection criteria")
print("="*80)

SLIDING WINDOW CORRELATION ANALYSIS - BY SUBPOPULATION
MULTIPLE TRIAL FILTERING CONDITIONS
STYLE: MARKERS + ERROR BARS
Subject: zarya
Sessions: 8
Filter types: ['small_headings', 'high_pdw_small', 'errors_zeros', 'low_pdw_small']
Areas: ['MST', 'VPS']

STEP 1: Loading Neural Features and Categorizing Units
Loaded regular_RT: 1353 units
Loaded tuning_RT: 1353 units
Loaded regular (best window): 1353 units

  Categorizing for ves...
    Total valid units: 1066
      both: 256
      regular_RT_only: 340
      tuning_RT_only: 341
      best_window_only: 129

  Categorizing for vis...
    Total valid units: 1108
      both: 378
      regular_RT_only: 328
      tuning_RT_only: 291
      best_window_only: 111

  Categorizing for comb...
    Total valid units: 1075
      both: 295
      regular_RT_only: 318
      tuning_RT_only: 329
      best_window_only: 133

STEP 2: Loading Partial Correlation Data (All Filter Types)
20250306 - small_headings: ✓
20250306 - high_pdw_small: ✓
20250306 - error

### partail corr as fucntion fo time

In [13]:
# ============================================================================
# STEP 4.5: Modified Aggregation Function - ONLY Significant Units (FIXED)
# WITH FILTER TYPE SUPPORT
# ============================================================================

def aggregate_valid_units_across_sessions_SIGNIFICANT(dates, area, alignment, modality, 
                                                      valid_units, filter_type,
                                                      align_signs=True, alpha=0.05):
    """
    Aggregate partial correlations across sessions for a specific filter type.
    NEW: Only include units that are significant (p < alpha) in EITHER heading OR choice
    at each timepoint.
    
    Parameters:
    -----------
    dates : list
        List of dates to aggregate
    area : str
        Brain area to filter (e.g., 'MST', 'VPS')
    alignment : str
        Alignment type
    modality : str
        Modality type
    valid_units : DataFrame
        Valid units for this modality
    filter_type : str
        Trial filtering type (e.g., 'small_headings', 'high_pdw_small')
    align_signs : bool
        Whether to align signs based on heading preference
    alpha : float
        Significance threshold (default: 0.05)
    """
    import ast
    
    condition = modality_to_condition[modality]
    
    all_heading_corrs = []
    all_choice_corrs = []
    time_axis = None
    
    units_used_count = 0
    
    for date in dates:
        date_str = str(date)
        # Load the appropriate filter-specific file
        pkl_file = partialcorr_dir / f"{subject}_{date_str}_all_partialcorr_centered_{filter_type}.pkl"
        
        if not pkl_file.exists():
            continue
        
        results = PartialCorrelationAnalyzer.load_results(pkl_file)
        
        if alignment not in results:
            continue
        
        if condition not in results[alignment]['conditions']:
            continue
        
        cond_data = results[alignment]['conditions'][condition]
        
        # Get data
        heading_corrs = cond_data['heading_corrs']  # time x neurons
        choice_corrs = cond_data['choice_corrs']
        
        # Get p-values
        heading_pvals = cond_data.get('heading_pvals', None)
        choice_pvals = cond_data.get('choice_pvals', None)
        
        if heading_pvals is None or choice_pvals is None:
            print(f"    Warning: No p-values found for {date_str} - {filter_type}, skipping significance filtering")
            heading_pvals = np.zeros_like(heading_corrs)
            choice_pvals = np.zeros_like(choice_corrs)
        
        unit_ids = results[alignment]['unit_ids']
        unit_areas = results[alignment]['unit_areas']
        
        if time_axis is None:
            time_axis = cond_data['time_axis']
        
        # Get valid units for this session
        session_valid = valid_units[valid_units['date'] == date]
        
        if len(session_valid) == 0:
            continue
        
        # Match units
        session_heading = []
        session_choice = []
        
        for _, unit_row in session_valid.iterrows():
            unit_id = unit_row['unit_id']
            
            if unit_id not in unit_ids:
                continue
            
            unit_idx = unit_ids.index(unit_id)
            
            # Check if this unit belongs to the requested area
            if unit_areas[unit_idx] != area:
                continue
            
            units_used_count += 1
            
            h_corr = heading_corrs[:, unit_idx]  # time series
            c_corr = choice_corrs[:, unit_idx]
            h_pval = heading_pvals[:, unit_idx]
            c_pval = choice_pvals[:, unit_idx]
            
            # Apply significance mask at each timepoint
            sig_mask = (h_pval < alpha) | (c_pval < alpha)
            
            # Set non-significant timepoints to NaN
            h_corr_filtered = h_corr.copy()
            c_corr_filtered = c_corr.copy()
            
            h_corr_filtered[~sig_mask] = np.nan
            c_corr_filtered[~sig_mask] = np.nan
            
            # Sign alignment if requested
            if align_signs:
                sig_h_values = h_corr_filtered[~np.isnan(h_corr_filtered)]
                
                if len(sig_h_values) > 0:
                    mean_h = np.nanmean(sig_h_values)
                    
                    if mean_h < 0:
                        h_corr_filtered = -h_corr_filtered
                        c_corr_filtered = -c_corr_filtered
            
            session_heading.append(h_corr_filtered)
            session_choice.append(c_corr_filtered)
        
        if len(session_heading) > 0:
            all_heading_corrs.extend(session_heading)
            all_choice_corrs.extend(session_choice)
    
    if len(all_heading_corrs) == 0:
        return None
    
    # Stack as (time, neurons)
    all_heading_corrs = np.array(all_heading_corrs).T
    all_choice_corrs = np.array(all_choice_corrs).T
    
    # Compute mean and SEM
    heading_mean = np.nanmean(all_heading_corrs, axis=1)
    heading_sem = np.nanstd(all_heading_corrs, axis=1) / np.sqrt(np.sum(~np.isnan(all_heading_corrs), axis=1))
    
    choice_mean = np.nanmean(all_choice_corrs, axis=1)
    choice_sem = np.nanstd(all_choice_corrs, axis=1) / np.sqrt(np.sum(~np.isnan(all_choice_corrs), axis=1))
    
    # Compute R²
    heading_r2_all = all_heading_corrs ** 2
    choice_r2_all = all_choice_corrs ** 2
    
    heading_r2_mean = np.nanmean(heading_r2_all, axis=1)
    heading_r2_sem = np.nanstd(heading_r2_all, axis=1) / np.sqrt(np.sum(~np.isnan(heading_r2_all), axis=1))
    
    choice_r2_mean = np.nanmean(choice_r2_all, axis=1)
    choice_r2_sem = np.nanstd(choice_r2_all, axis=1) / np.sqrt(np.sum(~np.isnan(choice_r2_all), axis=1))
    
    # Count units per timepoint
    n_units_per_timepoint = np.sum(~np.isnan(all_heading_corrs), axis=1)
    
    n_sessions = len(dates)
    total_units = all_heading_corrs.shape[1]
    
    return {
        'time_axis': time_axis,
        'heading_mean': heading_mean,
        'heading_sem': heading_sem,
        'heading_r2_mean': heading_r2_mean,
        'heading_r2_sem': heading_r2_sem,
        'choice_mean': choice_mean,
        'choice_sem': choice_sem,
        'choice_r2_mean': choice_r2_mean,
        'choice_r2_sem': choice_r2_sem,
        'n_sessions': n_sessions,
        'n_units_per_timepoint': n_units_per_timepoint,
        'total_units': total_units,
    }


# ============================================================================
# STEP 4.6: DIAGNOSTIC - Check unit distribution (per filter type)
# ============================================================================
print("\n" + "="*80)
print("DIAGNOSTIC: Checking unit distribution by area and modality")
print("="*80)

for modality in modalities:
    print(f"\n{modality_names[modality]}:")
    valid_units = valid_units_by_modality[modality]
    
    for area in areas_to_plot:
        area_units = valid_units[valid_units['area'] == area]
        
        n_units = len(area_units)
        n_sessions = area_units['date'].nunique() if n_units > 0 else 0
        
        print(f"  {area}: {n_units} units across {n_sessions} sessions")
        
        if n_units > 0:
            sample_units = area_units['unit_id'].head(3).tolist()
            print(f"    Sample units: {sample_units}")
            dates_list = sorted(area_units['date'].unique())
            print(f"    Dates ({len(dates_list)}): {dates_list[:3]}{'...' if len(dates_list) > 3 else ''}")

print("="*80)


# ============================================================================
# STEP 4.7: Define Color Schemes
# ============================================================================

# Define heading colors by modality
heading_colors = {
    'ves': "#00000000",      # Blue
    'vis': "#a02c2c",          # Green
    'comb': "#2741d6"         # Red
}

# Define choice color (same for all modalities)
choice_color = '#ff7f0e'  # Orange


# ============================================================================
# STEP 5: Create Plots (WITH FILTER TYPES)
# ============================================================================
print("\n" + "="*80)
print("STEP 5: Creating Plots (ONLY Significant Units, All Filter Types)")
print("="*80)

# Define filter types
filter_types = ['small_headings', 'high_pdw_small', 'errors_zeros', 'low_pdw_small']
filter_descriptions = {
    'small_headings': 'All trials with small headings (-10° < h < 10°)',
    'high_pdw_small': 'High PDW (confident) + small headings',
    'errors_zeros': 'Error trials + zero heading trials',
    'low_pdw_small': 'Low PDW (unconfident) + small headings'
}

saved_figures = []

for area in areas_to_plot:
    for filter_type in filter_types:
        print(f"\n{'='*80}")
        print(f"Creating plots for {area} - {filter_type}")
        print(f"Description: {filter_descriptions[filter_type]}")
        print(f"{'='*80}")
        
        # Create two figures: correlations and R²
        fig_corr, axes_corr = plt.subplots(3, 3, figsize=(20, 15))
        fig_r2, axes_r2 = plt.subplots(3, 3, figsize=(20, 15))
        
        has_any_data = False
        
        for row, modality in enumerate(modalities):
            valid_units = valid_units_by_modality[modality]
            
            # Filter for this area
            area_valid_units = valid_units[valid_units['area'] == area]
            
            print(f"\n  {modality_names[modality]}: {len(area_valid_units)} {area} units")
            
            for col, alignment in enumerate(alignments):
                ax_corr = axes_corr[row, col]
                ax_r2 = axes_r2[row, col]
                
                print(f"    Processing {alignment}...", end=" ")
                
                # Get aggregated data for this filter type
                data = aggregate_valid_units_across_sessions_SIGNIFICANT(
                    dates, area, alignment, modality, valid_units, filter_type,
                    align_signs=True, alpha=0.05
                )
                
                if data is None:
                    ax_corr.text(0.5, 0.5, f'No {area} data\n({filter_type})', 
                               ha='center', va='center',
                               transform=ax_corr.transAxes, fontsize=14, color='gray')
                    ax_r2.text(0.5, 0.5, f'No {area} data\n({filter_type})', 
                              ha='center', va='center',
                              transform=ax_r2.transAxes, fontsize=14, color='gray')
                    
                    if row == 0:
                        ax_corr.set_title(alignment_names[alignment], fontsize=13, fontweight='bold')
                        ax_r2.set_title(alignment_names[alignment], fontsize=13, fontweight='bold')
                    
                    print("No data")
                    continue
                
                has_any_data = True
                
                time_ms = data['time_axis'] * 1000
                n_sess = data['n_sessions']
                n_units_per_time = data['n_units_per_timepoint']
                total_units = data['total_units']
                
                print(f"✓ {total_units} units, {n_sess} sessions")
                print(f"      Time: [{time_ms[0]:.0f}, {time_ms[-1]:.0f}]ms, "
                      f"Units/timepoint: {np.nanmin(n_units_per_time):.0f}-{np.nanmax(n_units_per_time):.0f}")
                
                heading_color = heading_colors[modality]
                
                # ==================== Plot Correlations ====================
                
                ax_corr.errorbar(time_ms, data['heading_mean'],
                               yerr=data['heading_sem'],
                               color=heading_color,
                               linewidth=2,
                               linestyle='-',
                               marker='o',
                               markersize=4,
                               markerfacecolor=heading_color,
                               markeredgecolor='white',
                               markeredgewidth=0.5,
                               capsize=2,
                               capthick=1,
                               elinewidth=1,
                               label=f'Heading (n={total_units})',
                               alpha=0.8,
                               zorder=3)
                
                ax_corr.errorbar(time_ms, data['choice_mean'],
                               yerr=data['choice_sem'],
                               color=choice_color,
                               linewidth=2,
                               linestyle='-',
                               marker='s',
                               markersize=4,
                               markerfacecolor=choice_color,
                               markeredgecolor='white',
                               markeredgewidth=0.5,
                               capsize=2,
                               capthick=1,
                               elinewidth=1,
                               label=f'Choice (n={total_units})',
                               alpha=0.8,
                               zorder=3)
                
                # Reference lines
                ax_corr.axhline(0, color='gray', linestyle='--', alpha=0.4, linewidth=1, zorder=1)
                ax_corr.axvline(0, color='gray', linestyle='-', alpha=0.4, linewidth=1.5, zorder=1)
                
                # Add alignment annotation
                if alignment == 'stimOn':
                    if np.any(time_ms < 0):
                        ax_corr.axvspan(time_ms[time_ms < 0][0], 0, 
                                       alpha=0.1, color='blue', 
                                       label='Pre-stim', zorder=0)
                    ax_corr.text(0.02, 0.98, 't=0: stimulus onset', 
                               transform=ax_corr.transAxes,
                               fontsize=9, va='top', ha='left',
                               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
                elif alignment == 'saccOnset':
                    ax_corr.text(0.02, 0.98, 't=0: saccade onset', 
                               transform=ax_corr.transAxes,
                               fontsize=9, va='top', ha='left',
                               bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
                elif alignment == 'postTargHold':
                    ax_corr.text(0.02, 0.98, 't=0: target hold', 
                               transform=ax_corr.transAxes,
                               fontsize=9, va='top', ha='left',
                               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
                
                # Add unit count text
                n_units_median = np.nanmedian(n_units_per_time)
                ax_corr.text(0.98, 0.98, 
                           f'Total: {total_units} units\n'
                           f'Per timepoint:\n'
                           f'Median: {n_units_median:.0f}\n'
                           f'Range: {np.nanmin(n_units_per_time):.0f}-{np.nanmax(n_units_per_time):.0f}',
                           transform=ax_corr.transAxes,
                           fontsize=8, va='top', ha='right',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                # Labels
                if row == 2:
                    ax_corr.set_xlabel('Time (ms)', fontsize=11, fontweight='bold')
                if col == 0:
                    ax_corr.set_ylabel('Partial Correlation (r)\n[Sig in EITHER]', fontsize=11, fontweight='bold')
                
                if row == 0:
                    ax_corr.set_title(alignment_names[alignment], fontsize=13, fontweight='bold')
                
                if col == 2:
                    ax_corr.text(1.05, 0.5, modality_names[modality],
                               transform=ax_corr.transAxes, rotation=270,
                               fontsize=13, fontweight='bold', va='center')
                
                if row == 0 and col == 2:
                    ax_corr.legend(loc='upper right', fontsize=9, framealpha=0.9)
                
                ax_corr.grid(True, alpha=0.25, linestyle=':', linewidth=0.5)
                ax_corr.set_ylim([-0.4, 0.4])
                ax_corr.tick_params(labelsize=10)
                
                # Baseline text
                if alignment == 'stimOn' and np.any(time_ms < 0):
                    pre_stim_mask = time_ms < 0
                    baseline_h = np.nanmean(data['heading_mean'][pre_stim_mask])
                    baseline_c = np.nanmean(data['choice_mean'][pre_stim_mask])
                    ax_corr.text(0.02, 0.02, 
                               f'Pre-stim baseline:\nH={baseline_h:.3f}, C={baseline_c:.3f}',
                               transform=ax_corr.transAxes,
                               fontsize=8, va='bottom', ha='left',
                               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                # ==================== Plot R² ====================
                
                ax_r2.errorbar(time_ms, data['heading_r2_mean'],
                             yerr=data['heading_r2_sem'],
                             color=heading_color,
                             linewidth=2,
                             linestyle='-',
                             marker='o',
                             markersize=4,
                             markerfacecolor=heading_color,
                             markeredgecolor='white',
                             markeredgewidth=0.5,
                             capsize=2,
                             capthick=1,
                             elinewidth=1,
                             label=f'Heading (n={total_units})',
                             alpha=0.8,
                             zorder=3)
                
                ax_r2.errorbar(time_ms, data['choice_r2_mean'],
                             yerr=data['choice_r2_sem'],
                             color=choice_color,
                             linewidth=2,
                             linestyle='-',
                             marker='s',
                             markersize=4,
                             markerfacecolor=choice_color,
                             markeredgecolor='white',
                             markeredgewidth=0.5,
                             capsize=2,
                             capthick=1,
                             elinewidth=1,
                             label=f'Choice (n={total_units})',
                             alpha=0.8,
                             zorder=3)
                
                ax_r2.axhline(0, color='gray', linestyle='--', alpha=0.4, linewidth=1, zorder=1)
                ax_r2.axvline(0, color='gray', linestyle='-', alpha=0.4, linewidth=1.5, zorder=1)
                
                if alignment == 'stimOn':
                    if np.any(time_ms < 0):
                        ax_r2.axvspan(time_ms[time_ms < 0][0], 0, 
                                     alpha=0.1, color='blue', zorder=0)
                    ax_r2.text(0.02, 0.98, 't=0: stimulus onset', 
                             transform=ax_r2.transAxes,
                             fontsize=9, va='top', ha='left',
                             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
                elif alignment == 'saccOnset':
                    ax_r2.text(0.02, 0.98, 't=0: saccade onset', 
                             transform=ax_r2.transAxes,
                             fontsize=9, va='top', ha='left',
                             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
                elif alignment == 'postTargHold':
                    ax_r2.text(0.02, 0.98, 't=0: target hold', 
                             transform=ax_r2.transAxes,
                             fontsize=9, va='top', ha='left',
                             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
                
                ax_r2.text(0.98, 0.98, 
                         f'Total: {total_units} units\n'
                         f'Per timepoint:\n'
                         f'Median: {n_units_median:.0f}',
                         transform=ax_r2.transAxes,
                         fontsize=8, va='top', ha='right',
                         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
                
                if row == 2:
                    ax_r2.set_xlabel('Time (ms)', fontsize=11, fontweight='bold')
                if col == 0:
                    ax_r2.set_ylabel('R² (Variance Explained)\n[Sig in EITHER]', fontsize=11, fontweight='bold')
                
                if row == 0:
                    ax_r2.set_title(alignment_names[alignment], fontsize=13, fontweight='bold')
                
                if col == 2:
                    ax_r2.text(1.05, 0.5, modality_names[modality],
                             transform=ax_r2.transAxes, rotation=270,
                             fontsize=13, fontweight='bold', va='center')
                
                if row == 0 and col == 2:
                    ax_r2.legend(loc='upper right', fontsize=9, framealpha=0.9)
                
                ax_r2.grid(True, alpha=0.25, linestyle=':', linewidth=0.5)
                ax_r2.set_ylim([0.0, 0.4])
                ax_r2.tick_params(labelsize=10)
        
        # Only save if we have data
        if not has_any_data:
            print(f"\n  ⚠ No data for {area} - {filter_type}, skipping save")
            plt.close(fig_corr)
            plt.close(fig_r2)
            continue
        
        # Super titles
        fig_corr.suptitle(f'{area}: Partial Correlations - {filter_type.upper()}\n'
                         f'{filter_descriptions[filter_type]}\n'
                         f'ONLY Significant Units in EITHER (p<0.05)',
                         fontsize=16, fontweight='bold', y=0.995)
        
        fig_r2.suptitle(f'{area}: R² Values - {filter_type.upper()}\n'
                       f'{filter_descriptions[filter_type]}\n'
                       f'ONLY Significant Units in EITHER (p<0.05)',
                       fontsize=16, fontweight='bold', y=0.995)
        
        fig_corr.tight_layout(rect=[0, 0, 1, 0.98])
        fig_r2.tight_layout(rect=[0, 0, 1, 0.98])
        
        # Save with filter type in filename
        save_path_corr = partialcorr_dir / f'{subject}_{area}_partial_correlations_SIG_EITHER_{filter_type}.png'
        save_path_r2 = partialcorr_dir / f'{subject}_{area}_r2_values_SIG_EITHER_{filter_type}.png'
        
        fig_corr.savefig(save_path_corr, dpi=300, bbox_inches='tight')
        fig_r2.savefig(save_path_r2, dpi=300, bbox_inches='tight')
        
        saved_figures.append(save_path_corr)
        saved_figures.append(save_path_r2)
        
        print(f"\n  ✓ Saved: {save_path_corr.name}")
        print(f"  ✓ Saved: {save_path_r2.name}")
        
        plt.close(fig_corr)
        plt.close(fig_r2)

print("\n" + "="*80)
print("✅ PLOTTING COMPLETE!")
print("="*80)
print(f"\nGenerated {len(saved_figures)} figures total")
print(f"({len(saved_figures)//2} filter types × 2 areas × 2 plot types)")
print("\nFigures created:")
for fig_path in saved_figures:
    print(f"  - {fig_path.name}")
print("\nKey features:")
print("  ✓ Separate figures for each filter type")
print("  ✓ Area-specific filtering")
print("  ✓ Only significant units (p<0.05 in EITHER) at each timepoint")
print("  ✓ Number of contributing units tracked")
print("="*80)


DIAGNOSTIC: Checking unit distribution by area and modality

Vestibular:
  MST: 303 units across 8 sessions
    Sample units: [86, 93, 94]
    Dates (8): [20250306, 20250411, 20250417]...
  VPS: 262 units across 8 sessions
    Sample units: [545, 564, 569]
    Dates (8): [20250306, 20250411, 20250417]...

Visual High Coh:
  MST: 388 units across 8 sessions
    Sample units: [93, 94, 95]
    Dates (8): [20250306, 20250411, 20250417]...
  VPS: 285 units across 8 sessions
    Sample units: [556, 560, 564]
    Dates (8): [20250306, 20250411, 20250417]...

Combined High Coh:
  MST: 318 units across 8 sessions
    Sample units: [86, 94, 102]
    Dates (8): [20250306, 20250411, 20250417]...
  VPS: 259 units across 8 sessions
    Sample units: [552, 558, 560]
    Dates (8): [20250306, 20250411, 20250417]...

STEP 5: Creating Plots (ONLY Significant Units, All Filter Types)

Creating plots for MST - small_headings
Description: All trials with small headings (-10° < h < 10°)

  Vestibular: 303 

### sanity check

In [14]:
# Test on NULL data (pure noise)
def test_null_data():
    """Test if collinearity alone creates negative correlation"""
    np.random.seed(42)
    n_trials = 200
    n_neurons = 50
    
    # Real task structure
    heading = np.random.choice([-10, -5, -2.5, 0, 2.5, 5, 10], n_trials)
    choice = (heading > 0).astype(float)  # Perfect correlation
    choice += np.random.randn(n_trials) * 0.1  # Add tiny noise
    pdw = np.random.randn(n_trials)
    
    # Neural activity = PURE NOISE (no real heading or choice signal!)
    neural = np.random.randn(n_trials, n_neurons)
    
    behavior = {'heading': heading, 'choice': choice, 'PDW': pdw}
    
    analyzer = PartialCorrelationAnalyzer('test', 'test')
    h_corr, _ = analyzer.compute_heading_partial_correlation(neural, behavior)
    c_corr, _ = analyzer.compute_choice_partial_correlation(neural, behavior)
    
    # Correlation between the two partial correlations
    valid = ~(np.isnan(h_corr) | np.isnan(c_corr))
    r_null = pearsonr(h_corr[valid], c_corr[valid])[0]
    
    print(f"\n{'='*60}")
    print(f"NULL DATA TEST (pure noise neurons)")
    print(f"{'='*60}")
    print(f"Heading-choice correlation in task: {pearsonr(heading, choice)[0]:.3f}")
    print(f"Correlation between heading and choice partial corrs: {r_null:.3f}")
    print(f"")
    if r_null < -0.3:
        print(f"⚠️  NEGATIVE correlation even with pure noise!")
        print(f"⚠️  This means the method produces artifacts due to collinearity")
    else:
        print(f"✅ Near-zero correlation with noise (method is OK)")
    print(f"{'='*60}\n")
    
    return r_null

# Run this test
test_null_data()

Initialized with bin_size=20ms
  Window: 10 bins (200ms)
  Step: 1 bins (20ms)


AttributeError: 'PartialCorrelationAnalyzer' object has no attribute 'compute_heading_partial_correlation'